In [17]:
%pip install -q -U albumentations albucore timm

Note: you may need to restart the kernel to use updated packages.


In [18]:
import os, gc, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
for _i in range(torch.cuda.device_count()):
    print(f"  cuda:{_i}:", torch.cuda.get_device_name(_i))

Torch: 2.10.0+cu128
CUDA available: True
CUDA device count: 2
  cuda:0: Tesla T4
  cuda:1: Tesla T4


In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

import shutil
import zipfile

# ============================================================
# ValaisCD-paths
# ============================================================

VALAIS_INPUT_BASE = Path("/kaggle/input/valais-cd-dataset")

VALAIS_SCRATCH = Path(
    "/kaggle/temp" if Path("/kaggle/temp").exists() else "/tmp"
) / "valais_cd"

# 'dev' is the Brig region, used by the dataset authors only to train the
# label-pruning model. It is not part of the train/val/test protocol, so it is
# never extracted.
VALAIS_SPLITS = ("train", "val", "test")

VALAIS_SUBDIRS = ("2017", "2023", "labels")


def _is_split_dir(directory):
    """True if `directory` directly holds 2017/, 2023/ and labels/."""
    return all((directory / sub).is_dir() for sub in VALAIS_SUBDIRS)


def find_split_dir(base, split, max_depth=5):
    """Locate the directory holding one split's three image folders.

    Breadth-first, so the shallowest match wins. A candidate counts only if the
    split name appears somewhere in its path -- otherwise the search for 'test'
    would happily return the 'train' folder, which is a mislabelled-experiment
    class of bug rather than a crash.
    """
    if not base.exists():
        return None
    stack = [(base, 0)]
    while stack:
        directory, depth = stack.pop(0)
        if _is_split_dir(directory):
            parts = [part.lower() for part in directory.parts]
            if split in parts:
                return directory
        if depth < max_depth:
            try:
                stack.extend(
                    (p, depth + 1)
                    for p in sorted(directory.iterdir())
                    if p.is_dir()
                )
            except (PermissionError, OSError):
                continue
    return None


def resolve_splits(base):
    """{split: directory} for every split, or None entries for what is missing."""
    return {split: find_split_dir(base, split) for split in VALAIS_SPLITS}


def extract_valais_zips(base, destination):
    """Expand <split>.zip for every split in VALAIS_SPLITS into `destination`.

    Each archive already carries its split name as the top-level entry
    (train/2017/0.tif and so on), so they all extract into the same root. A
    split whose target folder already exists is skipped, which makes re-running
    this cell cheap instead of a 6.5 GB no-op.
    """
    archives = {}
    for split in VALAIS_SPLITS:
        found = sorted(base.rglob(split + ".zip"))
        archives[split] = found[0] if found else None

    missing = [s for s, p in archives.items() if p is None]
    if missing:
        raise FileNotFoundError(
            "Neither an extracted ValaisCD tree nor the archives "
            + str([m + ".zip" for m in missing])
            + " were found under "
            + str(base)
            + ".\nAttach the Kaggle dataset 'aparupghosh/valais-cd-dataset' "
            "to this notebook."
        )

    destination.mkdir(parents=True, exist_ok=True)

    for split, archive in archives.items():
        target = destination / split
        if all((target / sub).is_dir() for sub in VALAIS_SUBDIRS):
            print("  " + split + ": already extracted")
            continue
        size_gb = archive.stat().st_size / 1e9
        print("  " + split + ": extracting " + archive.name
              + " (" + format(size_gb, ".2f") + " GB) ...")
        # Extract into a .partial directory and rename, so an interrupted
        # session cannot leave a half-populated split that the skip-check
        # above would then accept as complete.
        staging = destination / ("." + split + ".partial")
        if staging.exists():
            shutil.rmtree(staging)
        with zipfile.ZipFile(archive) as handle:
            handle.extractall(staging)
        produced = staging / split if (staging / split).is_dir() else staging
        if target.exists():
            shutil.rmtree(target)
        produced.rename(target)
        if staging.exists():
            shutil.rmtree(staging, ignore_errors=True)

    return destination


DATA_ROOT = VALAIS_INPUT_BASE

VALAIS_SPLIT_DIRS = resolve_splits(DATA_ROOT)

if any(v is None for v in VALAIS_SPLIT_DIRS.values()):
    _absent = [k for k, v in VALAIS_SPLIT_DIRS.items() if v is None]
    print("Splits not found as directories under /kaggle/input:", _absent)
    print("Falling back to expanding the archives.")
    DATA_ROOT = extract_valais_zips(VALAIS_INPUT_BASE, VALAIS_SCRATCH)
    VALAIS_SPLIT_DIRS = resolve_splits(DATA_ROOT)
    _absent = [k for k, v in VALAIS_SPLIT_DIRS.items() if v is None]
    if _absent:
        raise FileNotFoundError(
            "Extraction finished but these splits still have no "
            + str(VALAIS_SUBDIRS)
            + " directory under " + str(DATA_ROOT) + ": " + str(_absent)
        )

OUT_DIR = Path("/kaggle/working/ValaisCD_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# DATASET VERSION
# ============================================================
VALAIS_VERSION = "5000"

VALAIS_LIST_TEMPLATE = "test_imgs_filtered_{version}_2.pkl"


# ============================================================
# TRAINING KNOBS
# ============================================================

IMG_SIZE = 256          
BATCH_SIZE = 8
NUM_WORKERS = 4        
EPOCHS = 60
LR = 5e-5
WD = 1e-5

CHANGE_TARGET_FRAC = 0.30
TRAIN_MULTIPLIER = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device      :", DEVICE)
print("Data root   :", DATA_ROOT)
for _split, _dir in VALAIS_SPLIT_DIRS.items():
    print(f"  {_split:<6}: {_dir}")
print("Version     :", VALAIS_VERSION or "full (unpruned)")
print("Output dir  :", OUT_DIR)

Device      : cuda
Data root   : /kaggle/input/valais-cd-dataset
  train : /kaggle/input/valais-cd-dataset/train/train
  val   : /kaggle/input/valais-cd-dataset/val/val
  test  : /kaggle/input/valais-cd-dataset/test/test
Version     : 5000
Output dir  : /kaggle/working/ValaisCD_results


In [ ]:
import pickle

# ============================================================
# BUILD THE SPLIT INDICES
# ============================================================
def _load_pruned_names(split_dir, version):
    if version is None:
        return None

    filename = VALAIS_LIST_TEMPLATE.format(version=version)
    candidates = [
        split_dir / filename,
        split_dir.parent / filename,
    ]
    list_path = next((p for p in candidates if p.exists()), None)

    if list_path is None:
        raise FileNotFoundError(
            "Pruned file list '" + filename + "' not found. Looked in:\n"
            + "\n".join("  " + str(p) for p in candidates)
            + "\nSet VALAIS_VERSION = None to use the full unpruned split, or "
            "pick one of the shipped versions (1000 / 2000 / 5000 / 10000)."
        )

    with open(list_path, "rb") as handle:
        entries = pickle.load(handle)
    return [str(e).replace("\\", "/").split("/")[-1] for e in entries]


def build_index(split, version=None):
    # Resolved in the paths cell, because the on-disk nesting differs between a
    # Kaggle-expanded dataset and a locally unzipped one.
    split_dir = VALAIS_SPLIT_DIRS[split]

    names = _load_pruned_names(split_dir, version)

    if names is None:
        names = sorted(
            p.name
            for p in (split_dir / "2017").iterdir()
            if p.suffix.lower() in (".tif", ".tiff", ".png", ".jpg")
        )

    rows = []
    dropped = 0
    for name in names:
        image1 = split_dir / "2017" / name
        image2 = split_dir / "2023" / name
        label = split_dir / "labels" / name
        if not (image1.exists() and image2.exists() and label.exists()):
            dropped += 1
            continue
        rows.append(
            {
                "name": name,
                "image1": str(image1),
                "image2": str(image2),
                "label": str(label),
                "label1": str(label),
                "label2": str(label),
            }
        )

    if dropped:
        print("  " + split + ": skipped " + str(dropped)
              + " incomplete samples")

    if not rows:
        raise RuntimeError("No usable samples found for split '" + split + "'")

    return pd.DataFrame(rows)


# ============================================================
# CHANGE STATISTICS
# ============================================================
# Two numbers per sample: whether it contains any change, and its positive
# pixel count. The first drives the training oversampling, the second is what
# justifies the loss weights and the low threshold grid, so it is worth
# reporting rather than assuming.
#
# Reading a few thousand 256x256 masks takes seconds, but it happens on every
# cell re-run, so the result is cached next to the other outputs.


def change_stats(df, split, version):
    tag = str(version) if version is not None else "full"
    cache = OUT_DIR / ("valais_change_" + split + "_" + tag + ".csv")

    if cache.exists():
        cached = pd.read_csv(cache)
        if len(cached) == len(df) and (cached["name"] == df["name"]).all():
            return cached["positive"].to_numpy()

    positives = np.zeros(len(df), dtype=np.int64)
    for i, path in enumerate(tqdm(df["label"].tolist(),
                                  desc="scan " + split, leave=False)):
        mask = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        if mask is None:
            raise FileNotFoundError(path)
        if mask.ndim == 3:
            mask = mask[..., 0]
        positives[i] = int((mask > 0).sum())

    pd.DataFrame({"name": df["name"], "positive": positives}).to_csv(
        cache, index=False
    )
    return positives


train_df = build_index("train", VALAIS_VERSION)
val_df = build_index("val", VALAIS_VERSION)
test_df = build_index("test", VALAIS_VERSION)

train_pos = change_stats(train_df, "train", VALAIS_VERSION)
val_pos = change_stats(val_df, "val", VALAIS_VERSION)
test_pos = change_stats(test_df, "test", VALAIS_VERSION)

train_df["positive"] = train_pos
val_df["positive"] = val_pos
test_df["positive"] = test_pos

PIXELS_PER_IMAGE = IMG_SIZE * IMG_SIZE

print("=" * 70)
print("VALAISCD SPLITS  (version: "
      + (VALAIS_VERSION or "full unpruned") + ")")
print("=" * 70)
print(f"{'split':<8}{'images':>8}{'changed':>9}{'changed%':>10}{'pos px%':>10}")
for split_name, frame in (("train", train_df), ("test", test_df),
                          ("val", val_df)):
    positive = frame["positive"].to_numpy()
    changed = int((positive > 0).sum())
    print(f"{split_name:<8}{len(frame):>8}{changed:>9}"
          f"{100.0 * changed / len(frame):>9.1f}%"
          f"{100.0 * positive.sum() / (len(frame) * PIXELS_PER_IMAGE):>9.3f}%")


# ============================================================
# CHANGE OVERSAMPLING FOR TRAINING
# ============================================================
# Repeat the changed rows until they make up CHANGE_TARGET_FRAC of the training
# index. Solving  c*r / (u + c*r) = f  for the repeat factor r gives
# r = f*u / (c*(1-f)), clamped to >= 1 so a target below the natural rate is a
# no-op rather than a silent downsample.

_changed_mask = train_df["positive"].to_numpy() > 0
_n_changed = int(_changed_mask.sum())
_n_unchanged = int(len(train_df) - _n_changed)

if _n_changed == 0:
    raise RuntimeError(
        "The training split contains no changed samples. Check that the "
        "labels are being read as {0,1} and not thresholded at 127."
    )

CHANGE_REPEAT = max(
    1,
    int(round(
        CHANGE_TARGET_FRAC * _n_unchanged
        / (_n_changed * (1.0 - CHANGE_TARGET_FRAC))
    )),
)

train_index_df = pd.concat(
    [train_df] + [train_df[_changed_mask]] * (CHANGE_REPEAT - 1),
    ignore_index=True,
)

_achieved = float((train_index_df["positive"] > 0).mean())

print()
print("changed rows repeated  :", CHANGE_REPEAT, "x")
print("train index size       :", len(train_index_df),
      "(from", len(train_df), "unique)")
print("changed fraction       : "
      + format(100.0 * _achieved, ".1f") + "%  (target "
      + format(100.0 * CHANGE_TARGET_FRAC, ".0f") + "%)")
print("effective pos-pixel %  : "
      + format(100.0 * train_index_df["positive"].sum()
               / (len(train_index_df) * PIXELS_PER_IMAGE), ".3f") + "%")

train_df.head()

scan test:   0%|          | 0/500 [00:00<?, ?it/s]

scan val:   0%|          | 0/1000 [00:00<?, ?it/s]

VALAISCD SPLITS  (version: 5000)
split     images  changed  changed%   pos px%
train       3500      175      5.0%    0.167%
test         500       25      5.0%    0.181%
val         1000       50      5.0%    0.248%

changed rows repeated  : 8 x
train index size       : 4725 (from 3500 unique)
changed fraction       : 29.6%  (target 30%)
effective pos-pixel %  : 0.990%


,name,image1,image2,label,label1,label2,positive
0,5095.tif,/kaggle/input/valais-cd-dataset/train/train/20...,/kaggle/input/valais-cd-dataset/train/train/20...,/kaggle/input/valais-cd-dataset/train/train/la...,/kaggle/input/valais-cd-dataset/train/train/la...,/kaggle/input/valais-cd-dataset/train/train/la...,0
1,1341.tif,/kaggle/input/valais-cd-dataset/train/train/20...,/kaggle/input/valais-cd-dataset/train/train/20...,/kaggle/input/valais-cd-dataset/train/train/la...,/kaggle/input/valais-cd-dataset/train/train/la...,/kaggle/input/valais-cd-dataset/train/train/la...,0
2,9104.tif,/kaggle/input/valais-cd-dataset/train/train/20...,/kaggle/input/valais-cd-dataset/train/train/20...,/kaggle/input/valais-cd-dataset/train/train/la...,/kaggle/input/valais-cd-dataset/train/train/la...,/kaggle/input/valais-cd-dataset/train/train/la...,0
3,10658.tif,/kaggle/input/valais-cd-dataset/train/train/20...,/kaggle/input/valais-cd-dataset/train/train/20...,/kaggle/input/valais-cd-dataset/train/train/la...,/kaggle/input/valais-cd-dataset/train/train/la...,/kaggle/input/valais-cd-dataset/train/train/la...,0
4,15.tif,/kaggle/input/valais-cd-dataset/train/train/20...,/kaggle/input/valais-cd-dataset/train/train/20...,/kaggle/input/valais-cd-dataset/train/train/la...,/kaggle/input/valais-cd-dataset/train/train/la...,/kaggle/input/valais-cd-dataset/train/train/la...,0


In [ ]:
def read_rgb(path):
    """Read a ValaisCD .tif as HxWx3 RGB uint8."""
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def read_change_mask(path):
    """Read a ValaisCD label as HxW uint8 in {0, 1}.

    ValaisCD stores masks with pixel values 0 and 1, not 0 and 255. The
    S2Looking version of this notebook thresholded at 127, which on this
    dataset returns an all-zero target for every sample: the loss still falls
    (predicting all-negative is optimal against an empty target) and every
    metric reads 0.0 with no error anywhere. IMREAD_UNCHANGED plus `> 0` is
    also correct for a 0/255 mask, so this is the safe reader either way.
    """
    mask = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if mask is None:
        raise FileNotFoundError(path)
    if mask.ndim == 3:
        mask = mask[..., 0]
    return (mask > 0).astype(np.uint8)
photo_tfms = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.6),
    A.HueSaturationValue(hue_shift_limit=6, sat_shift_limit=14,
                         val_shift_limit=10, p=0.4),
])


class ValaisCDDataset(Dataset):
    """ValaisCD pair -> (6-channel image, 3-channel mask stack)."""

    def __init__(self, df, transforms=None, photometric=False,
                 temporal_swap_p=0.0):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms
        self.photometric = photometric
        # ValaisCD labels are undirected differences between the 2017 and 2023
        # building footprints, so swapping the two dates leaves the target
        # valid. On a 3500-image train split that is free regularization; at
        # eval it is always 0.0.
        self.temporal_swap_p = temporal_swap_p

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        img1 = read_rgb(r["image1"])
        img2 = read_rgb(r["image2"])

        if self.temporal_swap_p and random.random() < self.temporal_swap_p:
            img1, img2 = img2, img1

        if self.photometric:
            img1 = photo_tfms(image=img1)["image"]    # two INDEPENDENT draws
            img2 = photo_tfms(image=img2)["image"]

        change = read_change_mask(r["label"])
        empty = np.zeros_like(change)

        # HxWx3 so albumentations transforms all three together.
        masks = np.stack([change, empty, empty], axis=-1)

        # HxWx6
        image = np.concatenate([img1, img2], axis=-1)

        if self.transforms is not None:
            out = self.transforms(image=image, mask=masks)
            image = out["image"]                        # tensor (6,H,W)
            masks = out["mask"]                         # tensor (H,W,3)
            masks = masks.permute(2, 0, 1).float()      # (3,H,W)
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
            masks = torch.from_numpy(masks).permute(2, 0, 1).float()
        return image, masks



In [ ]:
# ============================================================
# CREATE TRAIN / VAL / TEST DATASETS AND DATALOADERS
# Run this BEFORE the visualization cell
# ============================================================

required = [
    "ValaisCDDataset",
    "train_df",
    "val_df",
    "test_df",
    "train_index_df",
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(
        f"Run the earlier dataset-building cells first. Missing: {missing}"
    )


# ImageNet normalization for TWO RGB images = 6 channels
MEAN = [0.485, 0.456, 0.406] * 2
STD = [0.229, 0.224, 0.225] * 2


# ============================================================
# TRAIN TRANSFORMS
# ============================================================
train_tfms = A.Compose([
    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.15,
        rotate_limit=15,
        border_mode=cv2.BORDER_REFLECT_101,
        p=0.4,
    ),

    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),

    A.Normalize(mean=MEAN, std=STD, max_pixel_value=255.0),

    ToTensorV2(),
])


# ============================================================
# VALIDATION / TEST TRANSFORMS
# ============================================================

val_tfms = A.Compose([
    A.Normalize(mean=MEAN, std=STD, max_pixel_value=255.0),
    ToTensorV2(),
])


# ============================================================
# DATASETS
# ============================================================
# train_index_df is already change-oversampled; TRAIN_MULTIPLIER then sets how
# many augmented views of that index one epoch covers.

train_ds = ValaisCDDataset(
    pd.concat([train_index_df] * TRAIN_MULTIPLIER, ignore_index=True),
    transforms=train_tfms,
    photometric=True,
    temporal_swap_p=0.5,
)

val_ds = ValaisCDDataset(
    val_df,
    transforms=val_tfms,
    photometric=False,
    temporal_swap_p=0.0,
)

test_ds = ValaisCDDataset(
    test_df,
    transforms=val_tfms,
    photometric=False,
    temporal_swap_p=0.0,
)


# ============================================================
# DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)


# ============================================================
# VERIFY
# ============================================================

print("DATASETS CREATED")
print("Changed-row repeat  :", CHANGE_REPEAT, "x")
print("Train multiplier    :", TRAIN_MULTIPLIER)
print("Train samples       :", len(train_ds))
print("Val samples         :", len(val_ds))
print("Test samples        :", len(test_ds))
print("Steps/epoch @ bs", BATCH_SIZE, ":", len(train_loader))

img, masks = train_ds[0]

print("\nSample image shape :", tuple(img.shape))
print("Sample mask shape  :", tuple(masks.shape))
print("Mask channel sums  :",
      [float(masks[c].sum()) for c in range(masks.shape[0])])

# A non-degenerate target is the one thing worth asserting here: the 0/1 label
# encoding is exactly the kind of mistake that trains silently to zero. Probe a
# row the index cell already counted as changed, rather than the first N rows
# (which are ~95% unchanged and would make this check a coin flip).
_changed_rows = np.flatnonzero(
    train_ds.df["positive"].to_numpy() > 0
)
# Max over a few draws, because ShiftScaleRotate can legitimately push a small
# footprint out of frame and a single empty draw is not evidence of a bug.
_probe_sum = max(
    float(train_ds[int(row)][1][0].sum())
    for row in _changed_rows[:5]
)
print("\nPositive pixels in a known-changed sample:", _probe_sum)
if _probe_sum <= 0.0:
    raise RuntimeError(
        "A sample the index cell counted as changed came out of the dataset "
        "with an empty target. The label reader is wrong -- ValaisCD masks "
        "are {0, 1}, so a `> 127` threshold zeroes them."
    )

print("\ntrain_ds IS NOW READY")

DATASETS CREATED
Changed-row repeat  : 8 x
Train multiplier    : 2
Train samples       : 9450
Val samples         : 500
Test samples        : 1000
Steps/epoch @ bs 8 : 1181

Sample image shape : (6, 256, 256)
Sample mask shape  : (3, 256, 256)
Mask channel sums  : [0.0, 0.0, 0.0]

Positive pixels in a known-changed sample: 7812.0

train_ds IS NOW READY


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [ ]:
# ===================== QGMamba-CD: setup (cell A) =====================
# Kaggle T4x2, ValaisCD.
#
# Two input datasets are attached, deliberately separate:
#
#   /kaggle/input/qgmamba-cd-code       the qgmamba_cd package + configs
#   /kaggle/input/qgmamba-s2looking     the trained S2Looking checkpoint
#
# The code dataset is the one that gets updated when the repo changes (it is
# ~180 KB); the checkpoint dataset is ~900 MB and does not need re-uploading
# for a code edit. qgmamba-s2looking also ships an older copy of the package,
# so the code dataset is placed FIRST on sys.path.
# ================================================================

import sys
import random
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt


# ============================================================
# 1. PATHS (Kaggle datasets, read-only under /kaggle/input)
# ============================================================

QG_CODE_DIR = Path("/kaggle/input/qgmamba-cd-code")

QG_CKPT_DIR = Path("/kaggle/input/qgmamba-s2looking")

QG_CKPT_PATH = QG_CKPT_DIR / "qgmamba_s2looking_best_11.pt"

QG_PKG_DIR = QG_CODE_DIR                  # contains the qgmamba_cd/ package

# levir_final_config.yaml is the single config this project now runs on --
# every entry point defaults to it, and this notebook follows.
#
# NOTE: its MODEL section is NOT the one qgmamba_s2looking_best_11.pt was
# trained with (swin_tiny vs swin_small, corrected diffusion / edge fusion /
# refinement flags all set explicitly). The checkpoint is loaded strict=True,
# so a transfer run from that .pt will raise a key/shape error here. Use
# QG_INIT = "scratch" in cell C, or re-point QG_CKPT_PATH at a checkpoint
# trained under this config.
QG_CFG_PATH = str(QG_CODE_DIR / "configs" / "valais_cd_transfer.yaml")

# QGMamba is trained FROM SCRATCH on ValaisCD (ImageNet-pretrained encoder,
# everything else random). qgmamba_s2looking_best_11.pt cannot be used with
# levir_final_config.yaml: the checkpoint is swin_small + LegacyDiffusionDecoder,
# this config is swin_tiny + StochasticDiffusionDecoder (corrected_diffusion_
# enabled: true). Encoder depth AND the diffusion module differ, so no
# strict=False or key remap rescues it -- the two are mutually exclusive.
# To go back to transfer: point QG_CFG_PATH at configs/valais_cd_transfer.yaml
# and set QG_INIT = "s2looking".
QG_INIT = "s2looking"

print("=" * 70)
print("QGMAMBA PATH CHECK")
print("=" * 70)
print("Init      :", QG_INIT)
print("Checkpoint:", QG_CKPT_PATH, "(unused when QG_INIT == 'scratch')")
print("Package   :", QG_PKG_DIR / "qgmamba_cd")
print("Config    :", QG_CFG_PATH)

if QG_INIT != "scratch":
    assert QG_CKPT_PATH.exists(), (
        f"Checkpoint not found: {QG_CKPT_PATH}\n"
        "Attach the Kaggle dataset 'aparupghosh/qgmamba-s2looking' to this "
        "notebook."
    )

assert (QG_PKG_DIR / "qgmamba_cd" / "__init__.py").exists(), (
    f"qgmamba_cd package not found under {QG_PKG_DIR}\n"
    "Attach the Kaggle dataset 'aparupghosh/qgmamba-cd-code'. The .pt file "
    "contains weights; Python still needs the QGMamba model code to "
    "reconstruct the network."
)

assert Path(QG_CFG_PATH).exists(), (
    f"Config not found: {QG_CFG_PATH}\n"
    "QGMamba needs the architecture YAML to recreate the network before "
    "loading the .pt weights."
)

if str(QG_PKG_DIR) not in sys.path:
    sys.path.insert(0, str(QG_PKG_DIR))


# ============================================================
# 2. IMPORT QGMAMBA
# ============================================================

from qgmamba_cd.config import (
    load_config,
    EvalConfig
)

from qgmamba_cd.model import (
    QGMambaDiffCD,
    ModelEMA
)

from qgmamba_cd.checkpoint import (
    normalize_state_dict,
    save_checkpoint
)

from qgmamba_cd.losses import FullLoss

from qgmamba_cd.engine import (
    build_layerwise_parameter_groups
)

from qgmamba_cd.evaluation import (
    sweep_thresholds,
    evaluate_threshold,
    predict_probability_map,
    confusion_counts,
    metrics_from_confusion
)

from qgmamba_cd.distributed import (
    DistributedContext,
    seed_everything,
    configure_cuda
)

print("\nqgmamba_cd imports successful")
print("loaded from:", Path(sys.modules["qgmamba_cd"].__file__).parent)


# ============================================================
# 3. BUILD MODEL FUNCTION
# ============================================================

def build_qgmamba(pretrained=False):
    config = load_config(QG_CFG_PATH)
    config.model.pretrained = pretrained
    config.model.use_ddim_at_inference = True
    model = QGMambaDiffCD(config.model)
    assert getattr(model.diffusion, "use_ddim", False), "Refiner inactive — numbers would be coarse-only."
    return model

QG_CONFIG = load_config(QG_CFG_PATH)

QG_TILE = QG_CONFIG.model.input_size

assert QG_TILE == IMG_SIZE, (
    f"Model tile {QG_TILE} does not match the ValaisCD patch size {IMG_SIZE}. "
    "One ValaisCD patch is supposed to be exactly one model tile."
)

print("\nQGMamba tile size:", QG_TILE)


# ============================================================
# 4. DEVICE
# ============================================================

CONTEXT = DistributedContext(
    rank=0,
    local_rank=0,
    world_size=1,
    device=DEVICE
)

configure_cuda(torch.cuda.is_available())

QG_DTYPE = "float16" if torch.cuda.is_available() else "float32"

print("Device:", DEVICE)
print("Inference dtype:", QG_DTYPE)


# ============================================================
# 5. VALAISCD ADAPTER
# ============================================================
# The evaluation helpers in qgmamba_cd.evaluation expect
# (image_a, image_b, target, name) per item; the notebook datasets yield a
# 6-channel stack and a 3-channel mask. This splits them.

class ValaisPairDataset(torch.utils.data.Dataset):

    def __init__(self, df, transforms):

        self.inner = ValaisCDDataset(df, transforms)

        self.names = df["name"].tolist()

    def __len__(self):

        return len(self.inner)

    def __getitem__(self, index):

        image6, masks = self.inner[index]

        image_a = image6[:3]

        image_b = image6[3:]

        target = masks[0].cpu().numpy().astype(np.uint8)

        return image_a, image_b, target, self.names[index]

# ============================================================
# 6. VALIDATION / TEST SETS
# ============================================================

qg_val_ds = ValaisPairDataset(val_df, val_tfms)

qg_test_ds = ValaisPairDataset(test_df, val_tfms)

# Selection set: the FULL unpruned val (1716 patches, ~86 changed), used only
# to pick the threshold and the best-of-{single epoch, top-K average} model in
# cell D. qg_val_ds above (pruned, 500 patches / 25 changed) remains the split
# whose metrics get REPORTED. Selecting AND reporting on the same 25-changed
# pruned split is what let the old val F1 climb by chasing its own noise --
# see cell D for how the two are kept apart.
val_sel_df = build_index("val", None)

qg_val_sel_ds = ValaisPairDataset(val_sel_df, val_tfms)

val_sel_pos = change_stats(val_sel_df, "val", None)
val_sel_df["positive"] = val_sel_pos



# ============================================================
# 7. EVALUATION CONFIG
# ============================================================
# Taken from the YAML rather than restated here, so the tile geometry, the TTA
# flags and the threshold grid have exactly one definition. What the YAML sets
# for ValaisCD, and why:
#
#   tile_size = tile_stride = final_stride = 256
#       A ValaisCD patch IS one tile. On S2Looking these differed because a
#       1024x1024 image was tiled with overlap; here an overlapping stride
#       would produce the same single tile and only cost time.
#
#   thresholds spanning 0.05 .. 0.65
#       ValaisCD is ~0.18% positive pixels, five times sparser than
#       S2Looking. A model carrying an S2Looking prior into that is
#       systematically under-confident, so the useful operating points sit
#       well below 0.5 and the S2Looking grid (0.44 .. 0.80) would miss them
#       entirely, reporting a floor value as if it were the optimum.
#
#   final_tta = True
#       Flip TTA. Free accuracy at eval, and cheap on 256x256 inputs.

QG_EVAL = QG_CONFIG.eval

print("\nEVAL PROTOCOL")
print("  tile / stride / final :", QG_EVAL.tile_size, "/",
      QG_EVAL.tile_stride, "/", QG_EVAL.final_stride)
print("  batch_tiles           :", QG_EVAL.batch_tiles)
print("  tta / final_tta / d4  :", QG_EVAL.tta, "/", QG_EVAL.final_tta, "/",
      QG_EVAL.d4_tta)
print("  thresholds            :", QG_EVAL.thresholds)
print("  quick_val_images      :", QG_EVAL.quick_val_images)


# ============================================================
# 8. VISUALIZATION
# ============================================================

def denorm(image6):

    """
    Undo the ImageNet normalization applied by val_tfms / train_tfms.

    image6:
        tensor (6,H,W) -- 2017 RGB stacked on 2023 RGB

    Returns:
        pre, post -- each HxWx3 float in [0,1], ready for imshow
    """

    array = (
        image6
        .detach()
        .cpu()
        .float()
        .numpy()
        .transpose(1, 2, 0)
    )

    mean = np.asarray(MEAN, dtype=np.float32)

    std = np.asarray(STD, dtype=np.float32)

    array = np.clip(array * std + mean, 0.0, 1.0)

    return array[..., :3], array[..., 3:]


def qg_show(model, dataset, threshold, n=3, title="", prefer_changed=True,
            tta=False, d4_tta=False):

    """Show n samples as 2017 / 2023 / ground truth / prediction.

    prefer_changed biases the sample toward patches that actually contain
    change. On ValaisCD a uniform draw of 3 shows an empty ground truth about
    86% of the time, which makes the panel useless for judging anything.

    tta / d4_tta default to the previous hardcoded call (flip-only TTA
    off, D4 off) so any existing caller keeps today's panels; a caller
    reporting at a threshold calibrated WITH TTA must pass that TTA
    setting here too, or the panel shows a different operating point
    than the threshold printed under it.
    """

    model.eval()

    candidates = range(len(dataset))

    if prefer_changed:
        changed = [
            i for i in range(len(dataset))
            if int(dataset.inner.df.iloc[i].get("positive", 1)) > 0
        ]
        if changed:
            candidates = changed

    indices = random.sample(list(candidates), min(n, len(list(candidates))))

    fig, axes = plt.subplots(
        len(indices),
        4,
        figsize=(16, 4 * len(indices))
    )

    if len(indices) == 1:
        axes = axes[None, :]

    for row, i in enumerate(indices):

        image_a, image_b, target, name = dataset[i]

        probability = predict_probability_map(
            model,
            image_a,
            image_b,
            DEVICE,
            QG_TILE,
            QG_EVAL.tile_stride,
            QG_EVAL.batch_tiles,
            tta,
            True,
            QG_DTYPE,
            False,
            d4_tta
        )

        pre, post = denorm(torch.cat([image_a, image_b], dim=0))

        for ax in axes[row]:
            ax.axis("off")

        axes[row, 0].imshow(pre)
        axes[row, 0].set_title(f"2017 {name}")

        axes[row, 1].imshow(post)
        axes[row, 1].set_title("2023")

        axes[row, 2].imshow(target * 255, cmap="gray")
        axes[row, 2].set_title("GT change")

        prediction = (probability > threshold).astype(np.uint8)

        axes[row, 3].imshow(prediction * 255, cmap="gray")
        axes[row, 3].set_title(f"{title} pred @ {threshold:.3f}")

    plt.tight_layout()
    plt.show()


# ============================================================
# 9. CHECK CHECKPOINT CONTENT
# ============================================================

print("\nInspecting checkpoint structure...")

checkpoint = torch.load(
    QG_CKPT_PATH,
    map_location="cpu",
    weights_only=False
)

if isinstance(checkpoint, dict):
    print("Checkpoint top-level keys:", list(checkpoint.keys()))
else:
    print("Checkpoint type:", type(checkpoint))

del checkpoint


# ============================================================
# 10. FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("QGMAMBA-CD SETUP COMPLETE (ValaisCD)")
print("=" * 70)
print("Checkpoint :", QG_CKPT_PATH)
print("Config     :", QG_CFG_PATH)
print("Package    :", QG_PKG_DIR)
print("Tile       :", QG_TILE)
print("Val images (pruned, reported)   :", len(qg_val_ds),
      " changed:", int((test_df["positive"] > 0).sum()))
print("Val images (unpruned, selection):", len(qg_val_sel_ds),
      " changed:", int((val_sel_pos > 0).sum()))
print("Test images:", len(qg_test_ds))
print("Device     :", DEVICE)
print("=" * 70)
# --- EVAL HELPERS (extracted and tested by tests/test_eval_helpers.py) ---

def per_image_confusion(model, dataset, thresholds, config, device, amp=True,
                        amp_dtype="float16", indices=None, predict_fn=None,
                        desc="per-image"):
    """(n_images, n_thresholds, 4) int64 cube of (tp, tn, fp, fn) counts.

    One probability map per image, reused for every threshold. predict_fn
    defaults to the module-level predict_probability_map and exists so tests
    can inject a stub.
    """

    if predict_fn is None:
        predict_fn = predict_probability_map

    if indices is None:
        indices = range(len(dataset))

    threshold_array = np.asarray(thresholds, dtype=np.float32)

    cube = np.zeros((len(indices), len(threshold_array), 4), dtype=np.int64)

    for row, index in enumerate(tqdm(indices, desc=desc)):

        image_a, image_b, target, _ = dataset[index]

        # NOTE: unlike sweep_thresholds/evaluate_threshold in the package,
        # this ignores config.postprocess -- fine while postprocess: false
        # (the dataclass default and the YAML), but would silently diverge
        # from evaluate_threshold(..., final=True) if that flag ever flips.
        probability = predict_fn(
            model,
            image_a,
            image_b,
            device,
            config.tile_size,
            config.tile_stride,
            config.batch_tiles,
            config.tta,
            amp,
            amp_dtype,
            False,
            config.d4_tta
        )

        for threshold_index, threshold in enumerate(threshold_array):

            prediction = (probability > threshold).astype(np.uint8)

            cube[row, threshold_index] = confusion_counts(prediction, target)

    return cube


def threshold_rows(cube, thresholds):
    """[{'threshold': t, 'precision':..,'recall':..,'f1':..,'iou':..,'accuracy':..}]
    from the micro (summed-over-images) confusion at each threshold."""

    # Iterate the ORIGINAL threshold values (python floats), not a
    # np.float32 cast of them -- casting and back gives e.g. 0.075 ->
    # 0.07500000298023224, which then lands in pick_threshold_plateau's
    # return value, final_report['threshold'] and the saved checkpoint.
    threshold_list = list(thresholds)

    micro = cube.sum(axis=0)

    rows = []

    for threshold_index, threshold in enumerate(threshold_list):

        tp, tn, fp, fn = micro[threshold_index]

        rows.append({"threshold": float(threshold), **metrics_from_confusion(tp, tn, fp, fn)})

    return rows


def pick_threshold_plateau(rows, tolerance=0.99):
    """Median threshold of the CONTIGUOUS run of within-tolerance rows
    that contains the argmax (not every within-tolerance row anywhere on
    the grid).

    The argmax of a 50-object F1 curve is noise; the plateau centre is
    stable. Restricting to the contiguous run guards against a
    multi-lobed curve: taking the median of ALL within-tolerance rows can
    land in the valley between two lobes, an F1 below tolerance that the
    argmax rule was supposed to avoid returning. Returns one of the
    threshold values passed in, unchanged (a python float, not a float32
    round-trip -- see threshold_rows).
    """

    ordered = sorted(rows, key=lambda row: row["threshold"])

    best_f1 = max(row["f1"] for row in ordered)

    within = [row["f1"] >= tolerance * best_f1 for row in ordered]

    argmax_index = max(range(len(ordered)), key=lambda i: ordered[i]["f1"])

    lo = argmax_index
    while lo > 0 and within[lo - 1]:
        lo -= 1

    hi = argmax_index
    while hi < len(ordered) - 1 and within[hi + 1]:
        hi += 1

    plateau = ordered[lo:hi + 1]

    picked = plateau[len(plateau) // 2]

    assert picked["f1"] >= tolerance * best_f1, (
        "pick_threshold_plateau selected a threshold below tolerance"
    )

    return picked["threshold"]


def bootstrap_f1_ci(cube, threshold_index, iterations=2000, seed=0):
    """Resample IMAGES with replacement, recompute micro F1 each time.
    Returns {'f1': point estimate on the real sample, 'mean':.., 'lo':..,
    'hi':..} where lo/hi are the 2.5/97.5 percentiles."""

    counts = cube[:, threshold_index, :]

    n_images = counts.shape[0]

    point_f1 = metrics_from_confusion(*counts.sum(axis=0))["f1"]

    rng = np.random.default_rng(seed)

    samples = np.empty(iterations, dtype=np.float64)

    for i in range(iterations):

        sampled_indices = rng.integers(0, n_images, size=n_images)

        samples[i] = metrics_from_confusion(*counts[sampled_indices].sum(axis=0))["f1"]

    return {
        "f1": float(point_f1),
        "mean": float(samples.mean()),
        "lo": float(np.percentile(samples, 2.5)),
        "hi": float(np.percentile(samples, 97.5)),
    }


def macro_metrics(cube, threshold_index):
    """Mean per-image precision/recall/f1/iou over images that contain any
    ground-truth positive (tp+fn > 0). Returns the same dict shape as
    metrics_from_confusion, without 'accuracy'."""

    counts = cube[:, threshold_index, :]

    has_positive = (counts[:, 0] + counts[:, 3]) > 0

    per_image = [metrics_from_confusion(*row) for row in counts[has_positive]]

    return {
        key: float(np.mean([m[key] for m in per_image]))
        for key in ("precision", "recall", "f1", "iou")
    }

# --- END EVAL HELPERS ---


In [ ]:
# ===================== QGMamba-CD: zero-shot S2Looking -> ValaisCD (cell B) =====================


import gc
import json

_REQUIRED = [
    "build_qgmamba", "normalize_state_dict", "per_image_confusion",
    "threshold_rows", "pick_threshold_plateau", "bootstrap_f1_ci",
    "macro_metrics", "qg_val_ds", "qg_test_ds", "val_tfms",
    "ValaisPairDataset", "build_index",
    "QG_EVAL", "QG_CKPT_PATH", "DEVICE",
]
_missing = [x for x in _REQUIRED if x not in globals()]
if _missing:
    raise RuntimeError(f"Run the QGMamba setup cell (cell A) first. Missing: {_missing}")

# Unpruned "test" folder = physically Sierre = the paper's validation region.
# Used ONLY to select the threshold, never to report a final metric.
qg_test_sel_ds = ValaisPairDataset(build_index("test", None), val_tfms)

qg_ckpt = torch.load(QG_CKPT_PATH, map_location="cpu", weights_only=False)

zs_model = build_qgmamba(pretrained=False).to(DEVICE)

# S2Looking reported its EMA shadow as the eval weights (see cell C) -- prefer it.
zs_state = (
    qg_ckpt["ema_state"] if qg_ckpt.get("eval_weights") == "ema" and "ema_state" in qg_ckpt
    else qg_ckpt.get("model_state", qg_ckpt.get("state_dict"))
)
load_result = zs_model.load_state_dict(normalize_state_dict(zs_state), strict=True)
zs_model.eval()
print("checkpoint:", QG_CKPT_PATH.name,
      " missing keys:", len(load_result.missing_keys),
      " unexpected keys:", len(load_result.unexpected_keys))
del zs_state, qg_ckpt
gc.collect()

# Threshold selection on the unpruned "test" folder (physically Sierre/val).
sel_cube = per_image_confusion(
    zs_model, qg_test_sel_ds, QG_EVAL.thresholds, QG_EVAL, DEVICE, desc="zs-select-sierre"
)
sel_rows = threshold_rows(sel_cube, QG_EVAL.thresholds)
zs_threshold = pick_threshold_plateau(sel_rows)
zs_thr_idx = next(i for i, r in enumerate(sel_rows) if r["threshold"] == zs_threshold)
print("selected threshold:", round(zs_threshold, 4),
      " (Sierre-selection F1=", round(sel_rows[zs_thr_idx]["f1"], 4), ")")

# Report on both folders at that threshold. "val" (physically Martigny) is
# the number comparable to the paper's Table 2 supervised F1=0.46. "test"
# (physically Sierre) is the region the threshold was picked on, kept here
# only as a sanity check, not as the headline result.
zs_report = {}
for name, ds, town in (("val", qg_val_ds, "Martigny (paper test)"), ("test", qg_test_ds, "Sierre (paper val, selection region)")):
    cube = per_image_confusion(zs_model, ds, QG_EVAL.thresholds, QG_EVAL, DEVICE, desc=f"zs-{name}")
    rows = threshold_rows(cube, QG_EVAL.thresholds)
    idx = next(i for i, r in enumerate(rows) if r["threshold"] == zs_threshold)
    ci = bootstrap_f1_ci(cube, idx)
    macro = macro_metrics(cube, idx)
    zs_report[name] = {"town": town, "micro": rows[idx], "bootstrap_f1_ci": ci, "macro": macro}
    print(f"{name} ({town}): {rows[idx]}")
    print(f"  bootstrap_f1_CI=({ci['lo']:.4f},{ci['hi']:.4f})  macro={macro}")

print("ZERO-SHOT (S2Looking -> ValaisCD) SUMMARY, threshold selected on Sierre")
print(json.dumps(zs_report, indent=2))


## Kaggle session notes (T4 x2, ValaisCD)

**Scale.** ValaisCD is much smaller than S2Looking: the pruned `5000` version is
3500 train / 500 val / 1000 test 256×256 patches, against S2Looking's 3500
1024×1024 images. After change-oversampling (`CHANGE_REPEAT`) and
`TRAIN_MULTIPLIER = 2` an epoch is roughly 1.2k steps at batch 8, so expect
minutes per epoch rather than the 40–60 min S2Looking epochs. A single Kaggle
session covers the whole 60-epoch schedule.

**What actually had to change for this dataset**

| | S2Looking | ValaisCD |
|---|---|---|
| patch size | 1024², tiled to 256 | 256², one tile |
| label encoding | 0/255 | **0/1** |
| changed patches | ~100% | ~5% |
| positive pixels | ~1% | **~0.18%** |
| class balance handled by | forced non-empty crops | changed-row oversampling |
| threshold grid | 0.44 – 0.80 | **0.05 – 0.65** |

The two bolded rows are the ones that break silently. A `> 127` label threshold
returns an all-zero target on ValaisCD — the loss still falls, because
predicting all-negative is optimal against an empty target, and every metric
reads 0.0 with no error raised anywhere. And an all-zero prediction already
scores 99.8% pixel accuracy here, so a threshold grid that starts at 0.44 can
report its boundary value as "the optimum" while the real operating point sits
below it.

**Persistence.** `/kaggle/working` (and therefore `OUT_DIR`) survives only
within one session. Re-running the notebook resumes from
`qgmamba_valais_last.pt` only inside the same session. Between sessions: click
**Save Version** (which snapshots `/kaggle/working` as notebook output) or
download the checkpoints in `/kaggle/working/ValaisCD_results`, then copy them
back into `OUT_DIR` before restarting training.

**Extraction.** If Kaggle serves the dataset as four `.zip` archives rather
than an expanded tree, cell 2 expands them once into `/kaggle/temp/valais_cd`
(~6.5 GB, does not count against the output quota). That costs a few minutes at
the start of each session and is skipped on re-runs within a session.

In [ ]:
# ===================== QGMamba-CD: fine-tune on ValaisCD (cell C) =====================
# Kaggle T4x2 version (nn.DataParallel over both GPUs).
# Resumes from OUT_DIR/qgmamba_valais_last.pt if it already exists.
# Otherwise initialises from the S2Looking checkpoint in /kaggle/input and
# starts a FRESH ValaisCD run (see the transfer-reset block in section 5b:
# the S2Looking epoch counter, best-metric records and history describe a
# different dataset and must not be carried over).
#
# KAGGLE SESSION NOTES
# - ValaisCD epochs are cheap: ~1.2k steps at batch 8 on 256x256 inputs, a
#   few minutes each, so one session covers the whole schedule. This is the
#   opposite of the S2Looking run, where a single epoch took 40-60 min.
# - /kaggle/working (OUT_DIR) survives only within a session. Re-running the
#   notebook resumes from qgmamba_valais_last.pt ONLY inside the same
#   session. Between sessions, use "Save Version" so /kaggle/working is kept
#   as notebook output, or download the OUT_DIR checkpoints, and copy them
#   back into OUT_DIR before restarting.
# - QG_SNAPSHOT_EVERY = 0: /kaggle/working has a ~20 GB quota and each
#   checkpoint is ~900 MB, so epoch-tagged snapshots would blow the quota.
# - QG_INTRA_EPOCH_SAVE = 0: mid-epoch resume points bought insurance
#   against losing a 50-minute epoch. Losing a 3-minute one is not worth a
#   900 MB write.

import gc
import os
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from tqdm.auto import tqdm


# ============================================================
# 0. CLEAN OLD OBJECTS
# ============================================================

for _stale in (
    "model",
    "optimizer",
    "scheduler",
    "scaler",
    "zs_model",
    "qg_model",
    "qg_optimizer",
    "qg_scheduler",
    "qg_scaler",
    "qg_ema",
    "qg_train_step",
):
    if _stale in globals():
        del globals()[_stale]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# seed_everything() comes from qgmamba_cd.distributed and is normally bound by
# the QGMamba setup cell. Re-resolve it here so this cell also works when it is
# run on its own, or after a kernel restart, instead of raising NameError.

try:
    seed_everything

except NameError:

    try:
        from qgmamba_cd.distributed import seed_everything

    except Exception:

        def seed_everything(seed):

            random.seed(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)


seed_everything(SEED)


# ============================================================
# 1. REQUIRED OBJECT CHECK
# ============================================================

_REQUIRED = [
    "QG_CKPT_PATH",
    "QG_INIT",
    "QG_CFG_PATH",
    "QG_TILE",
    "QG_EVAL",
    "CONTEXT",
    "qg_val_ds",
    "train_ds",
    "CHANGE_REPEAT",
    "OUT_DIR",
    "DEVICE",
    "load_config",
    "QGMambaDiffCD",
    "ModelEMA",
    "FullLoss",
    "normalize_state_dict",
    "save_checkpoint",
    "build_layerwise_parameter_groups",
    "sweep_thresholds",
]

_missing = [
    x for x in _REQUIRED
    if x not in globals()
]

if _missing:
    raise RuntimeError(
        "Run the earlier setup/dataset cells first.\n"
        f"Missing: {_missing}"
    )


# ============================================================
# 2. CHECKPOINT PATHS (Kaggle)
# ============================================================
# QG_CKPT_PATH (the read-only /kaggle/input starting checkpoint) comes from
# the setup cell (cell A). Everything written during training goes to
# OUT_DIR under /kaggle/working.

QG_BEST_PATH = (
    OUT_DIR / "qgmamba_valais_best.pt"
)

QG_LAST_PATH = (
    OUT_DIR / "qgmamba_valais_last.pt"
)


# If we have already resumed training once,
# resume from the newest LAST checkpoint.
# Otherwise begin from the copied best checkpoint.

if QG_LAST_PATH.exists():

    QG_RESUME_PATH = QG_LAST_PATH

    print(
        "Using latest resumed checkpoint:"
    )

elif QG_INIT == "scratch":

    # From scratch on ValaisCD: the encoder keeps its ImageNet weights (set
    # below), everything else starts random. No source checkpoint exists,
    # because none matches levir_final_config.yaml -- see cell A.
    QG_RESUME_PATH = None

    print(
        "Training from scratch on ValaisCD "
        "(ImageNet-pretrained encoder, no source checkpoint)"
    )

else:

    QG_RESUME_PATH = QG_CKPT_PATH

    print(
        "Initialising from the S2Looking checkpoint "
        "(cross-dataset transfer):"
    )

if QG_RESUME_PATH is not None:

    print(QG_RESUME_PATH)

    assert QG_RESUME_PATH.exists(), (
        f"\nMissing checkpoint:\n{QG_RESUME_PATH}"
    )


# ============================================================
# 3. TRAINING CONFIG
# ============================================================

# ------------------------------------------------------------------------
# VALAISCD FINE-TUNE SCHEDULE
#
# This is a cross-dataset transfer, not a resume. The weights come from a
# converged S2Looking run; the data underneath them changes completely:
# 0.5 m/px orthorectified aerial imagery of Swiss alpine towns instead of
# 0.5-3 m satellite imagery, SwissTLM3D footprint differences instead of
# hand-drawn change masks, and a positive-pixel rate of ~0.18% instead of ~1%.
#
# So the schedule is a genuine cycle from a low peak, not the tail of an old
# one, and the epoch counter restarts at 1 (see the transfer-reset block in
# section 5b). Model and loss STRUCTURE come from valais_cd_transfer.yaml; the
# knobs below are optimiser, schedule, precision, EMA and checkpointing.
# ------------------------------------------------------------------------

# ValaisCD epochs are short (~1.2k steps at batch 8), so a schedule this long
# is affordable inside one Kaggle session.
QG_EPOCHS = 60

QG_BATCH = 8

# Peak LR. An order of magnitude under the 2e-4 used to train from ImageNet,
# because the encoder and the diffusion head already carry a usable
# change-detection prior; ten times the 2.5e-5 used for the S2Looking tail,
# because here the target distribution genuinely moves and the model has to
# travel, not settle.
QG_LR = 1.0e-4

QG_PATIENCE = 12

# EMA over ~2 epochs of warmup. Short, because there are only 60 epochs and
# the first ones carry the largest real improvement on a new domain.
QG_EMA_WARMUP = 2

QG_EMA_UPDATE_EVERY = 1

QG_EMA_REFERENCE_BATCH = 8

# Shorter horizon than the S2Looking 0.9999. At ~1.2k steps per epoch, 0.999
# averages roughly 1k steps (~1 epoch); 0.9999 would average 10k steps (~8
# epochs) and lag a run that is still moving fast.
QG_EMA_DECAY = 0.999

# BCE positive weight.
#
# Raw ValaisCD is ~0.18% positive pixels. The changed-row oversampling in the
# indexing cell already lifts the training stream to roughly 1%, i.e. back to
# the rate this checkpoint was trained at, so this does NOT also need to
# absorb the full imbalance -- doing both is how a model ends up predicting
# change everywhere and sweeping to a threshold of 0.9.
#
# 5.0 rather than the S2Looking 3.0 because even at 1% the remaining skew is
# larger, and because ValaisCD's positives are small compact footprints where
# a few false negatives cost proportionally more recall than on S2Looking's
# larger regions.
QG_POS_WEIGHT = 5.0

# Epochs at the start during which a non-improving epoch does NOT consume
# patience. On a fresh domain the first epochs are noise: EMA has not warmed
# up and the threshold sweep is still finding the operating point.
QG_FT_GRACE = 6

# Keep the N best epochs on disk for weight averaging after training.
QG_TOPK = 3

# None -> keep the YAML weight_decay. Zero-decay groups (norms, biases) are
# preserved either way.
QG_WEIGHT_DECAY = None

# A fresh cycle is mandatory here, not a preference: the restored scheduler
# state belongs to the S2Looking run.
QG_FRESH_SCHEDULE = True

QG_PCT_START = 0.10          # ~6 epochs of warmup on a 60-epoch cycle
QG_DIV_FACTOR = 10.0         # start at QG_LR / 10
QG_FINAL_DIV_FACTOR = 1.0e4  # anneal well below the starting LR

QG_LR_SCALE = 1.0            # uniform multiplier on the layerwise group LRs

# bfloat16 has fp32 dynamic range, so the GradScaler and its overflow/skip
# path disappear entirely. Falls back to float16 on pre-Ampere GPUs.
# Kaggle T4s are compute 7.5 and have no bf16 units. Do NOT gate on
# torch.cuda.is_bf16_supported(): since torch 2.6 it counts emulation and
# returns True on Turing, and the emulated bf16 kernels die with
# "CUDA error: misaligned address". Gate on compute capability >= 8 so a
# T4 takes the float16 + GradScaler branch below.
QG_PREFER_BF16 = True

# Checkpointing. Each checkpoint is ~900 MB against a ~20 GB /kaggle/working
# quota, so no epoch-tagged snapshots. Intra-epoch saves are OFF here (they
# were 500 on S2Looking): an epoch is minutes, so a session kill costs minutes,
# and paying a 900 MB write twice per epoch to insure that is a bad trade.
QG_SNAPSHOT_EVERY = 0        # epoch-tagged snapshot every N epochs, 0 = off
QG_INTRA_EPOCH_SAVE = 0      # mid-epoch save every N steps, 0 = off


# Inputs are a fixed 256x256, so let cuDNN pick and cache the fastest kernels.
# TF32 covers the fp32 ops that sit outside autocast.
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


QG_BF16 = (
    QG_PREFER_BF16
    and torch.cuda.is_available()
    and torch.cuda.get_device_capability()[0] >= 8
)

QG_AMP_DTYPE = (
    torch.bfloat16
    if QG_BF16
    else torch.float16
)

print(
    "AMP dtype       :",
    "bfloat16" if QG_BF16 else "float16"
)


# ============================================================
# 4. LOAD YAML CONFIG
# ============================================================

qg_cfg = load_config(
    QG_CFG_PATH
)

qg_cfg.train.epochs = QG_EPOCHS
qg_cfg.train.batch_size = QG_BATCH
qg_cfg.train.learning_rate = QG_LR

qg_cfg.train.amp_dtype = (
    "bfloat16"
    if QG_BF16
    else "float16"
)

# Single point of truth for the EMA horizon: ModelEMA in section 8 and
# qg_decay in section 16 both read this.
if QG_EMA_DECAY is not None:
    qg_cfg.train.ema_decay = QG_EMA_DECAY

if QG_WEIGHT_DECAY is not None:
    qg_cfg.train.weight_decay = QG_WEIGHT_DECAY

# Do not download pretrained encoder.
# Model weights come from checkpoint -- except from scratch, where the
# ImageNet-pretrained encoder IS the initialisation.
qg_cfg.model.pretrained = QG_RESUME_PATH is None


# ============================================================
# 5. LOAD CHECKPOINT
# ============================================================

if QG_RESUME_PATH is None:

    print("\nNo checkpoint to load (training from scratch).")

    _raw_ckpt = {}

else:

    print("\nLoading checkpoint...")

    _raw_ckpt = torch.load(
        QG_RESUME_PATH,
        map_location="cpu",
        weights_only=False
    )


# Handle full checkpoint OR bare state_dict
if QG_RESUME_PATH is None:

    qg_ckpt = {}

elif (
    isinstance(_raw_ckpt, dict)
    and (
        "model_state" in _raw_ckpt
        or "state_dict" in _raw_ckpt
        or "ema_state" in _raw_ckpt
    )
):

    qg_ckpt = _raw_ckpt

else:

    print(
        "Checkpoint appears to be a bare state_dict."
    )

    qg_ckpt = {
        "model_state": _raw_ckpt
    }


# ============================================================
# 5b. CROSS-DATASET TRANSFER RESET
# ============================================================
# When the starting point is the S2Looking checkpoint rather than a ValaisCD
# resume point, most of the bookkeeping inside it is about a different dataset
# and every piece of it is actively harmful here:
#
#   epoch                 ~140, so range(qg_start_epoch, QG_EPOCHS + 1) would
#                         be EMPTY and this cell would raise "already at epoch
#                         140" instead of training.
#   best_f1 / best_iou    S2Looking scores. Far above anything the first
#                         ValaisCD epochs will reach, so no checkpoint would
#                         ever be saved and early stopping would fire on
#                         schedule with nothing on disk.
#   history               S2Looking curves; plotting them against ValaisCD
#                         epoch numbers produces a chart that is simply wrong.
#   threshold             Selected on S2Looking val, where the positive rate is
#                         five times higher.
#   optimizer/scheduler   Adam moments and a OneCycle position from a finished
#                         run over a different loss surface.
#
# The weights themselves (model_state / ema_state) are exactly what we want and
# are left untouched. QG_FRESH_SCHEDULE already forces a new cycle; this makes
# the rest of the state consistent with that.

QG_IS_TRANSFER = QG_RESUME_PATH is not None and QG_RESUME_PATH != QG_LAST_PATH

if QG_IS_TRANSFER:

    _dropped = [
        key
        for key in (
            "epoch",
            "best_f1",
            "best_iou",
            "best_epoch",
            "history",
            "threshold",
            "val_metrics",
            "optimizer",
            "scheduler",
            "scaler",
        )
        if qg_ckpt.pop(key, None) is not None
    ]

    # The S2Looking run reported its EMA shadow as the eval weights, meaning
    # ema_state is the better of the two parameter sets. Section 7 initialises
    # the live model from model_state, so start BOTH from the EMA weights:
    # there is no reason to begin a transfer from the worse of two available
    # initialisations.
    if (
        qg_ckpt.get("eval_weights") == "ema"
        and "ema_state" in qg_ckpt
        and "model_state" in qg_ckpt
    ):
        import copy as _copy

        qg_ckpt["model_state"] = _copy.deepcopy(qg_ckpt["ema_state"])
        print("\nInitialising the live model from the checkpoint's EMA weights")

    print("\nCross-dataset transfer: S2Looking -> ValaisCD")
    print("  keeping   : model_state"
          + (", ema_state" if "ema_state" in qg_ckpt else ""))
    print("  discarding:", ", ".join(_dropped) or "(nothing)")
    print("  epoch counter restarts at 1")

elif QG_RESUME_PATH is None:

    print("\nFresh ValaisCD run from scratch; no checkpoint state to carry.")

else:

    print("\nResuming a ValaisCD run; checkpoint state kept intact.")


# ============================================================
# 6. DETERMINE START EPOCH
# ============================================================

_completed_epoch = int(
    qg_ckpt.get(
        "epoch",
        0
    ) or 0
)

qg_start_epoch = (
    _completed_epoch + 1
)


def _safe_float(value, default=-1.0):

    try:
        if value is None:
            return default

        return float(value)

    except Exception:
        return default


def _safe_int(value, default=0):

    try:
        if value is None:
            return default

        return int(value)

    except Exception:
        return default


print("\n" + "=" * 70)
print("RESUME INFORMATION")
print("=" * 70)

print(
    "checkpoint      :",
    QG_RESUME_PATH.name if QG_RESUME_PATH is not None else "(none, from scratch)"
)

print(
    "completed epoch :",
    _completed_epoch
)

print(
    "starting epoch  :",
    qg_start_epoch
)

print(
    "best IoU        :",
    _safe_float(
        qg_ckpt.get("best_iou"),
        float("nan")
    )
)

print(
    "best epoch      :",
    qg_ckpt.get(
        "best_epoch",
        "?"
    )
)

print(
    "eval weights    :",
    qg_ckpt.get(
        "eval_weights",
        "unknown"
    )
)


if qg_start_epoch > QG_EPOCHS:

    raise RuntimeError(
        f"Checkpoint is already at epoch "
        f"{_completed_epoch}.\n"
        f"QG_EPOCHS={QG_EPOCHS}.\n\n"
        f"Increase QG_EPOCHS above "
        f"{_completed_epoch} to continue training."
    )


# ============================================================
# 7. BUILD MODEL
# ============================================================

print("\nBuilding QGMamba...")

qg_model = QGMambaDiffCD(
    qg_cfg.model
).to(
    DEVICE
)


# Get correct model weights
if "model_state" in qg_ckpt:

    _model_sd = qg_ckpt[
        "model_state"
    ]

elif "state_dict" in qg_ckpt:

    _model_sd = qg_ckpt[
        "state_dict"
    ]

elif QG_RESUME_PATH is None:

    _model_sd = None

else:

    raise RuntimeError(
        "No model_state/state_dict found."
    )


if _model_sd is None:

    print("✓ Model left at its scratch initialisation")

else:

    _model_sd = normalize_state_dict(
        _model_sd
    )

    load_result = qg_model.load_state_dict(
        _model_sd,
        strict=True
    )

    print("✓ Model weights restored")

    print(
        "missing keys    :",
        len(load_result.missing_keys)
    )

    print(
        "unexpected keys :",
        len(load_result.unexpected_keys)
    )

del _model_sd


# ============================================================
# 8. RESTORE EMA
# ============================================================

qg_ema = ModelEMA(
    qg_model,
    decay=qg_cfg.train.ema_decay
)


if "ema_state" in qg_ckpt:

    _ema_sd = normalize_state_dict(
        qg_ckpt["ema_state"]
    )

    try:

        qg_ema.load_state_dict(
            _ema_sd
        )

    except (
        AttributeError,
        TypeError,
        RuntimeError
    ):

        qg_ema.ema.load_state_dict(
            _ema_sd,
            strict=True
        )

    print("✓ EMA shadow weights restored")

    del _ema_sd

else:

    print(
        "WARNING: checkpoint has no ema_state."
    )

    print(
        "EMA initialized from current model."
    )


# ============================================================
# 9. LOSS
# ============================================================

qg_criterion = FullLoss(
    qg_cfg.loss,
    pos_weight=QG_POS_WEIGHT
).to(
    DEVICE
)


# ============================================================
# 10. OPTIMIZER
# ============================================================

qg_param_groups, qg_max_lrs = (
    build_layerwise_parameter_groups(
        qg_model,
        qg_cfg
    )
)

# Uniform multiplier, so the layerwise decay ratios the repo assigned are
# preserved exactly -- only the absolute scale moves.
qg_max_lrs = [
    float(lr) * QG_LR_SCALE
    for lr in qg_max_lrs
]

print(
    f"layerwise peak LR: "
    f"{min(qg_max_lrs):.2e} .. {max(qg_max_lrs):.2e} "
    f"across {len(qg_max_lrs)} groups"
)

qg_optimizer = torch.optim.AdamW(
    qg_param_groups,
    lr=qg_cfg.train.learning_rate,
    weight_decay=qg_cfg.train.weight_decay
)


if "optimizer" in qg_ckpt:

    try:

        qg_optimizer.load_state_dict(
            qg_ckpt["optimizer"]
        )

        print(
            "✓ AdamW optimizer state restored"
        )

    except Exception as e:

        print(
            "WARNING: optimizer state could "
            "not be restored."
        )

        print(
            "Starting optimizer fresh:"
        )

        print(e)

else:

    print(
        "No optimizer state in checkpoint "
        "— optimizer starts fresh."
    )


# load_state_dict above restores the OLD weight_decay into every param group,
# so a changed QG_WEIGHT_DECAY would silently do nothing without this. Groups
# that were deliberately set to zero decay (norms, biases) stay at zero.
if QG_WEIGHT_DECAY is not None:

    for _group in qg_optimizer.param_groups:

        if _group.get("weight_decay", 0.0) > 0.0:

            _group["weight_decay"] = QG_WEIGHT_DECAY

    print(
        f"weight decay reapplied: {QG_WEIGHT_DECAY}"
    )


# ============================================================
# 11. TRAIN LOADER
# ============================================================

qg_train_loader = DataLoader(

    train_ds,

    batch_size=QG_BATCH,

    shuffle=True,

    num_workers=NUM_WORKERS,

    pin_memory=torch.cuda.is_available(),

    drop_last=True,

    # Each sample reads three 256x256 tifs -- roughly 1/80th of the S2Looking
    # sample -- so the loader is no longer the bottleneck. persistent_workers
    # still matters more here than it did there: epochs are short, so a
    # per-epoch worker respawn is a larger fraction of the epoch.
    persistent_workers=NUM_WORKERS > 0,

    prefetch_factor=(
        4
        if NUM_WORKERS > 0
        else None
    )
)


_steps_per_epoch = len(
    qg_train_loader
)

print(
    "\nTraining batches per epoch:",
    _steps_per_epoch
)


# ============================================================
# 12. SCHEDULER
# ============================================================

_sched_ckpt = qg_ckpt.get(
    "scheduler"
)

_ckpt_total = (
    _sched_ckpt.get(
        "total_steps"
    )
    if isinstance(
        _sched_ckpt,
        dict
    )
    else None
)

_planned_total = (
    _steps_per_epoch
    * QG_EPOCHS
)


# QG_FRESH_SCHEDULE forces the gentle-anneal branch below. Restoring the old
# OneCycle state would put the LR back on the original full-training curve.
if (
    not QG_FRESH_SCHEDULE
    and _sched_ckpt is not None
    and _ckpt_total == _planned_total
):

    qg_scheduler = (
        torch.optim.lr_scheduler.OneCycleLR(

            qg_optimizer,

            max_lr=qg_max_lrs,

            epochs=QG_EPOCHS,

            steps_per_epoch=
                _steps_per_epoch,

            pct_start=0.1,

            div_factor=25.0,

            final_div_factor=1000.0,

            anneal_strategy="cos"
        )
    )


    try:

        qg_scheduler.load_state_dict(
            _sched_ckpt
        )

        qg_step = int(
            _sched_ckpt.get(
                "last_epoch",
                (
                    qg_start_epoch - 1
                )
                * _steps_per_epoch
            )
        )

        print(
            f"✓ OneCycle resumed at "
            f"step {qg_step}/"
            f"{_planned_total}"
        )

    except Exception as e:

        print(
            "Scheduler restore failed:"
        )

        print(e)

        qg_step = (
            qg_start_epoch - 1
        ) * _steps_per_epoch


else:

    _remaining = (
        QG_EPOCHS
        - qg_start_epoch
        + 1
    )

    qg_scheduler = (
        torch.optim.lr_scheduler.OneCycleLR(

            qg_optimizer,

            max_lr=qg_max_lrs,

            epochs=_remaining,

            steps_per_epoch=
                _steps_per_epoch,

            pct_start=QG_PCT_START,

            div_factor=QG_DIV_FACTOR,

            final_div_factor=QG_FINAL_DIV_FACTOR,

            anneal_strategy="cos"
        )
    )

    qg_step = (
        qg_start_epoch - 1
    ) * _steps_per_epoch


    print(
        "Fresh fine-tune cycle over "
        f"{_remaining} remaining epochs."
    )

    print(
        f"  warmup    : "
        f"{QG_PCT_START * _remaining:.1f} epochs"
    )

    print(
        f"  LR path   : "
        f"{max(qg_max_lrs) / QG_DIV_FACTOR:.2e}"
        f" -> {max(qg_max_lrs):.2e}"
        f" -> {max(qg_max_lrs) / QG_FINAL_DIV_FACTOR:.2e}"
    )


# ============================================================
# 13. GRAD SCALER
# ============================================================

USE_AMP = torch.cuda.is_available()

# bfloat16 needs no loss scaling. With enabled=False the scale/unscale_/step/
# update calls in the loop below become pass-throughs, so the training loop
# itself is byte-identical either way.
qg_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP and not QG_BF16
)


if (
    "scaler" in qg_ckpt
    and USE_AMP
    and not QG_BF16
):

    try:

        qg_scaler.load_state_dict(
            qg_ckpt["scaler"]
        )

        print(
            "✓ AMP scaler restored"
        )

    except Exception as e:

        print(
            "WARNING: scaler state "
            "not restored:"
        )

        print(e)


# ============================================================
# 13b. MULTI-GPU TRAIN STEP (nn.DataParallel over 2x T4)
# ============================================================
# qg_model stays a BARE module on cuda:0: the optimizer parameter groups,
# gradient clipping, the EMA shadow and every checkpoint state_dict all read
# it directly, so saved checkpoints never grow a 'module.' prefix and stay
# loadable with strict=True through normalize_state_dict.
#
# Only this thin wrapper is replicated across GPUs. Its forward runs the
# model AND the full criterion, so the complete FullLoss (including the
# batch-flattened Lovasz term) is computed per replica on its own shard and
# only the per-replica scalar losses are gathered -- the six model outputs
# (refined, coarse, auxiliary tuple, edge logits, diffusion loss,
# orthogonal loss) never cross GPUs.
#
# torch.autocast is thread-local and nn.DataParallel runs each replica in
# its own thread, so autocast must be entered INSIDE forward. An autocast
# block around the qg_train_step(...) call in the training loop would
# silently not apply to the replica threads.

class QGTrainStep(nn.Module):

    def __init__(self, model, criterion, use_amp, amp_dtype):
        super().__init__()
        self.model = model
        self.criterion = criterion
        self.use_amp = use_amp
        self.amp_dtype = amp_dtype

    def forward(self, image_a, image_b, target):
        with torch.autocast(
            device_type="cuda",
            enabled=self.use_amp and image_a.is_cuda,
            dtype=self.amp_dtype,
        ):
            outputs = self.model(image_a, image_b, target)
            loss = self.criterion(*outputs, target)
        # (1,) per replica; DataParallel gathers these into an (n_gpus,)
        # vector on cuda:0, which the loop reduces with .mean().
        return loss.reshape(1)


qg_train_step = QGTrainStep(
    qg_model,
    qg_criterion,
    USE_AMP,
    QG_AMP_DTYPE,
)

QG_NUM_GPUS = torch.cuda.device_count()

if QG_NUM_GPUS > 1:

    qg_train_step = nn.DataParallel(qg_train_step)

    print(
        f"nn.DataParallel over {QG_NUM_GPUS} GPUs "
        f"(global batch {QG_BATCH} = "
        f"{QG_BATCH // QG_NUM_GPUS} per GPU)"
    )

else:

    print("Single-device training (no DataParallel)")


# ============================================================
# 14. RANDOM 256x256 CROP
# ============================================================

def random_crop_pair(
    images6,
    masks3
):

    """
    images6:
        (B,6,H,W)

    masks3:
        (B,3,H,W)

    Returns:
        image A
        image B
        all-change target
    """

    _, _, height, width = (
        images6.shape
    )


    if (
        height < QG_TILE
        or width < QG_TILE
    ):

        raise ValueError(
            f"Input size {height}x{width} "
            f"is smaller than QG_TILE="
            f"{QG_TILE}"
        )


    list_a = []
    list_b = []
    list_t = []


    for i in range(
        images6.shape[0]
    ):

        y = random.randint(
            0,
            height - QG_TILE
        )

        x = random.randint(
            0,
            width - QG_TILE
        )


        list_a.append(
            images6[
                i,
                :3,
                y:y + QG_TILE,
                x:x + QG_TILE
            ]
        )


        list_b.append(
            images6[
                i,
                3:,
                y:y + QG_TILE,
                x:x + QG_TILE
            ]
        )


        list_t.append(
            masks3[
                i,
                0:1,
                y:y + QG_TILE,
                x:x + QG_TILE
            ]
        )


    return (
        torch.stack(list_a),
        torch.stack(list_b),
        torch.stack(list_t)
    )


# ============================================================
# 15. HISTORY / BEST VALUES
# ============================================================

_hist_keys = (
    "epoch",
    "train_loss",
    "threshold",
    "val_precision",
    "val_recall",
    "val_f1",
    "val_iou",
    "eval_weights",
)


_existing_history = (
    qg_ckpt.get("history")
    or {}
)


qg_history = {

    k: list(
        _existing_history.get(
            k,
            []
        )
    )

    for k in _hist_keys
}


qg_best_iou = _safe_float(
    qg_ckpt.get("best_iou"),
    -1.0
)

qg_best_f1 = _safe_float(
    qg_ckpt.get("best_f1"),
    -1.0
)

qg_best_epoch = _safe_int(
    qg_ckpt.get("best_epoch"),
    0
)

qg_best_threshold = _safe_float(
    qg_ckpt.get("threshold"),
    0.5
)


# Recover best from history if necessary
if qg_history["val_iou"]:

    _i = int(
        np.argmax(
            qg_history[
                "val_iou"
            ]
        )
    )

    _history_best_iou = float(
        qg_history[
            "val_iou"
        ][_i]
    )


    if (
        _history_best_iou
        > qg_best_iou
    ):

        qg_best_iou = (
            _history_best_iou
        )

        qg_best_f1 = float(
            qg_history[
                "val_f1"
            ][_i]
        )

        qg_best_epoch = int(
            qg_history[
                "epoch"
            ][_i]
        )

        qg_best_threshold = float(
            qg_history[
                "threshold"
            ][_i]
        )


        print(
            "Best trackers corrected "
            "from history:"
        )

        print(
            f"IoU {qg_best_iou:.4f} "
            f"@ epoch "
            f"{qg_best_epoch}"
        )


# ============================================================
# 16. EMA DECAY
# ============================================================

qg_decay = (

    (
        qg_cfg.train.ema_decay
        ** (
            QG_BATCH
            / QG_EMA_REFERENCE_BATCH
        )
    )

    ** QG_EMA_UPDATE_EVERY
)


qg_stall = 0

# (val_iou, epoch, path) for the QG_TOPK best epochs, newest sort each epoch.
qg_topk = []


# ============================================================
# 16b. ATOMIC CHECKPOINT SAVE
# ============================================================

def qg_save(path, state):

    """
    Write to a .tmp file and rename into place.

    torch.save streams to disk, so a session kill or a full disk quota
    partway through a direct write leaves a truncated file where the good
    checkpoint used to be. os.replace is atomic on POSIX, so the destination
    is either the previous checkpoint or the complete new one, never a
    half-written one.

    Returns the file size in MB.
    """

    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    tmp = path.with_name(
        path.name + ".tmp"
    )

    save_checkpoint(
        tmp,
        state
    )

    os.replace(
        tmp,
        path
    )

    return path.stat().st_size / 1e6


def qg_state(epoch, threshold, val_metrics, use_ema):

    """Assemble the resume payload. Same keys the loader in section 5 reads."""

    return {

        "epoch":
            int(epoch),

        "model_state":
            qg_model.state_dict(),

        "ema_state":
            qg_ema.state_dict(),

        "optimizer":
            qg_optimizer.state_dict(),

        "scheduler":
            qg_scheduler.state_dict(),

        "scaler":
            qg_scaler.state_dict(),

        "threshold":
            threshold,

        "val_metrics":
            val_metrics,

        "best_f1":
            qg_best_f1,

        "best_epoch":
            qg_best_epoch,

        "best_iou":
            qg_best_iou,

        "history":
            qg_history,

        "eval_weights":
            (
                "ema"
                if use_ema
                else "model"
            ),

        "resumed_from":
            (
                str(QG_RESUME_PATH)
                if QG_RESUME_PATH is not None
                else "scratch"
            ),
    }


# Release original checkpoint copy
del qg_ckpt
del _raw_ckpt

gc.collect()


print("\n" + "=" * 70)

print(
    f"TRAINING EPOCHS "
    f"{qg_start_epoch}.."
    f"{QG_EPOCHS}"
)

print(
    f"Baseline best IoU: "
    f"{qg_best_iou:.4f}"
)

print("=" * 70)


# ============================================================
# 17. TRAIN
# ============================================================

for epoch in range(
    qg_start_epoch,
    QG_EPOCHS + 1
):

    # Puts qg_model and the criterion into train mode through the wrapper.
    qg_train_step.train()

    epoch_start = time.time()

    epoch_losses = []


    progress = tqdm(

        qg_train_loader,

        desc=(
            f"qgmamba "
            f"{epoch:02d}/"
            f"{QG_EPOCHS}"
        ),

        leave=False
    )


    for images6, masks3 in progress:


        image_a, image_b, target = (
            random_crop_pair(
                images6,
                masks3
            )
        )


        image_a = image_a.to(
            DEVICE,
            non_blocking=True
        )

        image_b = image_b.to(
            DEVICE,
            non_blocking=True
        )

        target = target.to(
            DEVICE,
            non_blocking=True
        )


        qg_optimizer.zero_grad(
            set_to_none=True
        )


        # autocast lives inside QGTrainStep.forward (thread-local; see
        # section 13b). Each replica returns its loss as shape (1,); the
        # gathered (n_gpus,) vector is averaged into one scalar here.
        loss = qg_train_step(
            image_a,
            image_b,
            target
        ).mean()


        if not torch.isfinite(loss):

            print(
                "WARNING: non-finite "
                "loss; skipping batch."
            )

            continue


        qg_scaler.scale(
            loss
        ).backward()


        qg_scaler.unscale_(
            qg_optimizer
        )


        torch.nn.utils.clip_grad_norm_(

            qg_model.parameters(),

            max_norm=
                qg_cfg.train.grad_clip
        )


        qg_scaler.step(
            qg_optimizer
        )

        qg_scaler.update()


        if (
            qg_scheduler.last_epoch
            < qg_scheduler.total_steps - 1
        ):

            qg_scheduler.step()


        qg_step += 1


        if (
            qg_step
            % QG_EMA_UPDATE_EVERY
            == 0
        ):

            qg_ema.update(
                qg_model,
                decay=qg_decay
            )


        # Optional mid-epoch resume point. Epochs run ~40-60 min on T4x2,
        # so a session kill otherwise costs the whole epoch. epoch-1 is
        # recorded so a resume re-runs this epoch from its start.
        if (
            QG_INTRA_EPOCH_SAVE
            and qg_step % QG_INTRA_EPOCH_SAVE == 0
        ):

            qg_save(
                QG_LAST_PATH,
                qg_state(
                    epoch - 1,
                    qg_best_threshold,
                    {},
                    True
                )
            )


        epoch_losses.append(
            float(
                loss.item()
            )
        )


        progress.set_postfix(

            loss=(
                f"{np.mean(epoch_losses):.4f}"
            )
        )


    # ========================================================
    # VALIDATE
    # ========================================================

    if not epoch_losses:

        print(
            f"Epoch {epoch}: "
            "no finite training batches."
        )

        continue


    use_ema = (
        epoch
        > QG_EMA_WARMUP
    )


    eval_model = (
        qg_ema.ema
        if use_ema
        else qg_model
    )


    threshold, val_metrics, _ = (
        sweep_thresholds(

            eval_model,

            qg_val_ds,

            QG_EVAL.thresholds,

            QG_EVAL,

            CONTEXT,

            amp=USE_AMP,

            amp_dtype="float16"
        )
    )


    threshold = float(
        threshold
    )


    # ========================================================
    # UPDATE HISTORY
    # ========================================================

    qg_history[
        "epoch"
    ].append(
        int(epoch)
    )


    qg_history[
        "train_loss"
    ].append(
        float(
            np.mean(
                epoch_losses
            )
        )
    )


    qg_history[
        "threshold"
    ].append(
        threshold
    )


    for key in (
        "precision",
        "recall",
        "f1",
        "iou"
    ):

        qg_history[
            f"val_{key}"
        ].append(
            float(
                val_metrics[key]
            )
        )


    qg_history[
        "eval_weights"
    ].append(
        "ema"
        if use_ema
        else "model"
    )


    # ========================================================
    # CHECK IMPROVEMENT FIRST
    # ========================================================

    improved = (
        float(
            val_metrics["iou"]
        )
        > qg_best_iou
    )


    if improved:

        qg_best_iou = float(
            val_metrics["iou"]
        )

        qg_best_f1 = float(
            val_metrics["f1"]
        )

        qg_best_epoch = int(
            epoch
        )

        qg_best_threshold = (
            threshold
        )

        qg_stall = 0

    else:

        qg_stall += 1


    # Raw -> EMA transition should
    # not consume patience.
    if (
        epoch
        == QG_EMA_WARMUP + 1
    ):

        qg_stall = 0


    # Nor should the re-convergence window after a distribution change.
    if (
        epoch
        < qg_start_epoch + QG_FT_GRACE
    ):

        qg_stall = 0


    # ========================================================
    # SAVE STATE
    # ========================================================

    state = qg_state(
        epoch,
        threshold,
        val_metrics,
        use_ema
    )


    # Always save latest state
    _last_mb = qg_save(
        QG_LAST_PATH,
        state
    )


    # Save best separately
    if improved:

        _best_mb = qg_save(
            QG_BEST_PATH,
            state
        )

        print(
            f"  saved best -> "
            f"{QG_BEST_PATH.name} "
            f"({_best_mb:.0f} MB)"
        )


    # Epoch-tagged snapshot, so a bad best/last is not the end of the run
    if (
        QG_SNAPSHOT_EVERY
        and epoch % QG_SNAPSHOT_EVERY == 0
    ):

        qg_save(
            OUT_DIR
            / f"qgmamba_valais_ep{epoch:03d}.pt",
            state
        )


    # ========================================================
    # TOP-K RETENTION FOR WEIGHT AVERAGING
    # ========================================================
    # A low-LR fine-tune keeps every late epoch inside one loss basin, so the
    # top epochs are points around a single minimum rather than different
    # solutions. Averaging them cancels the per-epoch gradient noise that no
    # single checkpoint can shed. Cell D builds and scores the average.

    _topk_path = (
        OUT_DIR
        / f"qgmamba_valais_top_{epoch:03d}.pt"
    )

    qg_topk.append(
        (
            float(val_metrics["iou"]),
            int(epoch),
            _topk_path
        )
    )

    qg_topk.sort(
        key=lambda t: -t[0]
    )


    _keep = qg_topk[:QG_TOPK]
    _drop = qg_topk[QG_TOPK:]


    if any(
        e == epoch
        for _, e, _ in _keep
    ):

        qg_save(
            _topk_path,
            state
        )


    for _, _, _p in _drop:

        if _p.exists():

            _p.unlink()


    qg_topk = _keep


    # ========================================================
    # STATUS
    # ========================================================

    print(

        f"qgmamba "
        f"{epoch:02d}/"
        f"{QG_EPOCHS}"

        f" | loss "
        f"{np.mean(epoch_losses):.4f}"

        f" | val IoU "
        f"{val_metrics['iou']:.4f}"

        f" F1 "
        f"{val_metrics['f1']:.4f}"

        f" @ thr "
        f"{threshold:.2f}"

        f" ({'ema' if use_ema else 'raw'})"

        f" | best IoU "
        f"{qg_best_iou:.4f}"

        f" (ep {qg_best_epoch})"

        f"{' -> BEST UPDATED' if improved else ''}"

        f" | "
        f"{time.time() - epoch_start:.0f}s"
    )


    # ========================================================
    # EARLY STOP
    # ========================================================

    if (
        qg_stall
        >= QG_PATIENCE
    ):

        print(
            f"Early stopping: "
            f"no val IoU improvement "
            f"for {QG_PATIENCE} epochs."
        )

        break


# ============================================================
# 18. SAVE HISTORY
# ============================================================

history_path = (
    OUT_DIR
    / "qgmamba_valais_history.csv"
)

pd.DataFrame(
    qg_history
).to_csv(
    history_path,
    index=False
)

# ========================================================
# STATUS — PRINT ALL METRICS
# ========================================================

print(
    f"qgmamba {epoch:02d}/{QG_EPOCHS}"
    f" | loss {np.mean(epoch_losses):.4f}"
    f" | Precision {val_metrics['precision']:.4f}"
    f" | Recall {val_metrics['recall']:.4f}"
    f" | F1 {val_metrics['f1']:.4f}"
    f" | IoU {val_metrics['iou']:.4f}"
    f" | thr {threshold:.2f}"
    f" | {'EMA' if use_ema else 'RAW'}"
    f" | best IoU {qg_best_iou:.4f}"
    f" (ep {qg_best_epoch})"
    f"{' -> BEST UPDATED' if improved else ''}"
    f" | {time.time() - epoch_start:.0f}s"
)


# ============================================================
# 19. CHECKPOINT VERIFICATION
# ============================================================

print("\n" + "=" * 70)
print("CHECKPOINTS ON DISK")
print("=" * 70)

for _p in (
    QG_BEST_PATH,
    QG_LAST_PATH
):

    if _p.exists():

        print(
            f"  {_p.name:38s} "
            f"{_p.stat().st_size / 1e6:8.1f} MB"
        )

    else:

        print(
            f"  MISSING: {_p}"
        )


for _p in sorted(
    OUT_DIR.glob(
        "qgmamba_valais_ep*.pt"
    )
):

    print(
        f"  {_p.name:38s} "
        f"{_p.stat().st_size / 1e6:8.1f} MB"
    )


print("\nTop-%d epochs retained for weight averaging:" % QG_TOPK)

for _iou, _ep, _p in qg_topk:

    print(
        f"  epoch {_ep:3d}  val IoU {_iou:.4f}  "
        f"{'present' if _p.exists() else 'MISSING'}"
    )


# Confirm the best checkpoint actually reloads before trusting the run
_verify = torch.load(
    QG_BEST_PATH,
    map_location="cpu",
    weights_only=False
)

print(
    f"\nbest checkpoint reloads OK: "
    f"epoch {_verify.get('epoch')}, "
    f"eval_weights {_verify.get('eval_weights')}, "
    f"keys {len(_verify)}"
)

del _verify
gc.collect()

print(
    f"\nbest val IoU {qg_best_iou:.4f} / "
    f"F1 {qg_best_f1:.4f} @ epoch {qg_best_epoch}, "
    f"threshold {qg_best_threshold:.3f}"
)

print(
    "\nRun the final-evaluation cell (cell D) next: overlap tiling at "
    "stride 128 plus D4 TTA is where the remaining points are."
)       

In [ ]:
# ===================== QGMamba-CD: final ValaisCD evaluation (cell D) =====================
# Loads the best-val-IoU checkpoint written by cell C, selects the operating
# threshold and the best-of-{single epoch, top-K average} model on the FULL
# unpruned val split (Sierre, 1716 patches, ~86 changed), then reports val and
# test metrics -- with bootstrap CIs, macro metrics and a test-oracle bound --
# at that threshold on the smaller pruned val split (500 patches, 25 changed)
# cell C actually validated against during training.
#
# FRAMING: train / val / test are three different Valais towns (Sion / Sierre
# / Martigny). A val -> test gap is therefore expected from cross-city domain
# shift alone, on top of whatever gap threshold transfer contributes -- the
# test-oracle bound computed below is what separates the two. Read a gap here
# as the split design working as intended, not as a bug.
import json
import gc

# ============================================================
# PRECONDITION CHECK
# ============================================================
# Everything below is built by the QGMamba setup cell (cell A). Without it this
# cell dies on a bare NameError deep in the body, so fail immediately and name
# what is missing. QG_EVAL is NOT rebuilt here on purpose: it fixes the tile
# size, stride, TTA flags and threshold grid, so a second definition that
# drifts from cell A would silently change the reported metrics.

_REQUIRED = [
    "build_qgmamba",
    "normalize_state_dict",
    "evaluate_threshold",
    "sweep_thresholds",
    "per_image_confusion",
    "threshold_rows",
    "pick_threshold_plateau",
    "bootstrap_f1_ci",
    "macro_metrics",
    "qg_show",
    "qg_val_ds",
    "qg_val_sel_ds",
    "qg_test_ds",
    "EvalConfig",
    "QG_TILE",
    "QG_EVAL",
    "CONTEXT",
    "DEVICE",
    "OUT_DIR",
]

_missing = [
    x for x in _REQUIRED
    if x not in globals()
]

if _missing:
    raise RuntimeError(
        "Run the QGMamba setup cell (cell A) before this one.\n"
        f"Missing: {_missing}"
    )


# ============================================================
# CHECKPOINT
# ============================================================
# Prefer the ValaisCD checkpoint cell C wrote. Falling back to the raw
# S2Looking weights keeps this cell runnable on its own, but it is a DIFFERENT
# experiment (zero-shot transfer, which is what cell B measures), so say so
# loudly rather than reporting it as a ValaisCD result.

QG_BEST_PATH = OUT_DIR / 'qgmamba_valais_best.pt'

if QG_BEST_PATH.exists():
    FINAL_SOURCE = 'valais_finetuned'
else:
    raise FileNotFoundError(
        f'No ValaisCD checkpoint at {QG_BEST_PATH}. Run cell C first.\n'
        'There is no fallback: the S2Looking checkpoint is a different '
        'architecture from levir_final_config.yaml and cannot be loaded.'
    )

qg_best = torch.load(QG_BEST_PATH, map_location='cpu', weights_only=False)
print(f"checkpoint: {QG_BEST_PATH.name}  source={FINAL_SOURCE}")
print(f"  epoch={qg_best.get('best_epoch', '?')}  "
      f"val IoU={qg_best.get('best_iou', float('nan')):.4f}  "
      f"eval_weights={qg_best.get('eval_weights', 'model')}")

final_model = build_qgmamba(pretrained=False)
final_state = (
    qg_best['ema_state']
    if qg_best.get('eval_weights') == 'ema' and 'ema_state' in qg_best
    else qg_best.get('model_state', qg_best.get('state_dict'))
)
final_model.load_state_dict(normalize_state_dict(final_state), strict=True)
final_model = final_model.to(DEVICE).eval()
del final_state
gc.collect()

# The strong protocol. On ValaisCD a patch is exactly one tile, so overlap
# tiling has nothing to overlap and the entire gain over QG_EVAL comes from D4
# TTA (8 views instead of 1). The threshold grid is a refinement of the YAML
# grid around the same low region -- 25 points over [0.05, 0.65] at 0.025 --
# because at 0.18% positive pixels the operating point sits far below 0.5 and
# the S2Looking grid ([0.40, 0.76]) would report a boundary value as the
# optimum. tile_stride == final_stride and tta == final_tta here, so the
# per_image_confusion cubes below (which always use the non-"final" fields)
# match evaluate_threshold(..., final=True) exactly -- there is no separate
# "quick" reading hiding inside FINAL_EVAL.
FINAL_EVAL = EvalConfig(
    tile_size=QG_TILE, tile_stride=QG_TILE, final_stride=QG_TILE,
    batch_tiles=16,
    tta=True, final_tta=True, d4_tta=True,
    thresholds=[round(0.05 + 0.025 * i, 4) for i in range(25)],
)

# ============================================================
# THRESHOLD + MODEL SELECTION (on the unpruned val, cube-based)
# ============================================================
# Selection now runs on qg_val_sel_ds (1716 patches, ~86 changed) instead of
# the pruned qg_val_ds (500 patches, 25 changed / ~50 change blobs) the old
# cell swept: 25 changed images is too few for an argmax threshold to be
# anything but noise, and that noise is exactly what let the old val F1 climb
# to a self-selected maximum. One probability map per image is computed once
# (per_image_confusion) and every grid threshold's confusion is derived from
# it -- at a real cost: selection moved from the 500-image pruned val to
# the 1716-image unpruned val, 3.4x the forward passes per candidate model
# and about 6.9x total across the two candidates (best-single-epoch and
# top-K average) scored in this cell. Justified by what it buys (a stable
# threshold instead of a 25-changed-image coin flip), but it is not free.
# pick_threshold_plateau replaces the raw argmax with the median of the
# thresholds within 1% of the best F1 -- the plateau centre, not whichever
# point happened to sit on top of the noise.

def _select_on_val(model, desc):
    cube = per_image_confusion(
        model, qg_val_sel_ds, FINAL_EVAL.thresholds, FINAL_EVAL, CONTEXT.device,
        amp=True, amp_dtype='float16', desc=desc)
    rows = threshold_rows(cube, FINAL_EVAL.thresholds)
    threshold = pick_threshold_plateau(rows)
    index = next(i for i, row in enumerate(rows) if row['threshold'] == threshold)
    return threshold, index, rows[index]['f1']


final_threshold, final_threshold_index, final_sel_f1 = _select_on_val(
    final_model, 'select best-single')

# ============================================================
# WEIGHT AVERAGING OVER THE TOP-K EPOCHS
# ============================================================
# Averaging the parameters of the best epochs of a low-LR fine-tune. No
# architecture or forward-pass change: the averaged tensors are loaded into
# the same model with strict=True. It is only a candidate here -- whichever
# of {best single epoch, averaged} wins on the unpruned val is the one scored
# on test, so this cannot quietly inflate the reported number.

def average_state_dicts(states):
    """Mean of float tensors; integer buffers taken from the first state."""
    keys = set(states[0])
    for s in states[1:]:
        if set(s) != keys:
            raise RuntimeError('checkpoint key mismatch, refusing to average')
    out = {}
    for k, v in states[0].items():
        if torch.is_floating_point(v):
            acc = torch.zeros_like(v, dtype=torch.float64)
            for s in states:
                acc += s[k].to(torch.float64)
            out[k] = (acc / len(states)).to(v.dtype)
        else:
            out[k] = v.clone()          # num_batches_tracked and friends
    return out


top_paths = sorted(OUT_DIR.glob('qgmamba_valais_top_*.pt'))
print(f"\ntop-K checkpoints found: {len(top_paths)}")

swa_model = None

if len(top_paths) >= 2:
    top_states, top_epochs = [], []
    for p in top_paths:
        ck = torch.load(p, map_location='cpu', weights_only=False)
        sd = ck['ema_state'] if ck.get('eval_weights') == 'ema' else ck['model_state']
        top_states.append(normalize_state_dict(sd))
        top_epochs.append(ck.get('epoch'))
        del ck
    print('averaging epochs:', top_epochs)

    swa_model = build_qgmamba(pretrained=False)
    swa_model.load_state_dict(average_state_dicts(top_states), strict=True)
    swa_model = swa_model.to(DEVICE).eval()
    del top_states
    gc.collect()

    swa_threshold, swa_threshold_index, swa_sel_f1 = _select_on_val(
        swa_model, 'select top-K average')

    print(f"val(unpruned) F1 @ plateau thr  "
          f"best-single {final_sel_f1:.4f} @ {final_threshold:.3f}  "
          f"averaged {swa_sel_f1:.4f} @ {swa_threshold:.3f}  "
          f"({swa_sel_f1 - final_sel_f1:+.4f})")

    # Selection on the unpruned val only.
    if swa_sel_f1 > final_sel_f1:
        print('-> averaged weights win on val (unpruned), scoring those on test')
        final_model = swa_model
        final_threshold, final_threshold_index = swa_threshold, swa_threshold_index
        SELECTED = 'topk_average'
        torch.save({'model_state': final_model.state_dict(),
                    'eval_weights': 'model', 'threshold': final_threshold,
                    'averaged_epochs': top_epochs,
                    'selection_metrics': {'f1': swa_sel_f1,
                                           'selection_set': 'val_unpruned_1716'}},
                   OUT_DIR / 'qgmamba_valais_swa.pt')
    else:
        print('-> best single epoch wins on val (unpruned), keeping it')
        SELECTED = 'best_single_epoch'
        del swa_model
        gc.collect()
else:
    print('not enough top-K checkpoints to average, using best single epoch')
    SELECTED = 'best_single_epoch'

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ============================================================
# REPORTED METRICS, WITH INTERVALS
# ============================================================
# The winning model, scored at its val-selected threshold, on the two splits
# that matter for the paper number (pruned val, the one training validated
# against) and the held-out test city. Both cubes are computed once here and
# reused for the micro metrics, the bootstrap CI and the macro metrics below
# -- no repeated inference passes.

val_cube = per_image_confusion(
    final_model, qg_val_ds, FINAL_EVAL.thresholds, FINAL_EVAL, CONTEXT.device,
    amp=True, amp_dtype='float16', desc='report val (pruned)')
test_cube = per_image_confusion(
    final_model, qg_test_ds, FINAL_EVAL.thresholds, FINAL_EVAL, CONTEXT.device,
    amp=True, amp_dtype='float16', desc='report test')

val_rows = threshold_rows(val_cube, FINAL_EVAL.thresholds)
test_rows = threshold_rows(test_cube, FINAL_EVAL.thresholds)

_metric_keys = ('precision', 'recall', 'f1', 'iou', 'accuracy')
final_val = {k: val_rows[final_threshold_index][k] for k in _metric_keys}
final_test = {k: test_rows[final_threshold_index][k] for k in _metric_keys}

val_ci = bootstrap_f1_ci(val_cube, final_threshold_index)
test_ci = bootstrap_f1_ci(test_cube, final_threshold_index)
val_macro = macro_metrics(val_cube, final_threshold_index)
test_macro = macro_metrics(test_cube, final_threshold_index)

print(f"\nval  (pruned 500)  F1 {val_ci['f1']:.4f}  95% CI [{val_ci['lo']:.4f}, {val_ci['hi']:.4f}]")
print(f"test               F1 {test_ci['f1']:.4f}  95% CI [{test_ci['lo']:.4f}, {test_ci['hi']:.4f}]")
print(f"val  macro precision/recall/f1/iou: "
      f"{val_macro['precision']:.4f}/{val_macro['recall']:.4f}/"
      f"{val_macro['f1']:.4f}/{val_macro['iou']:.4f}")
print(f"test macro precision/recall/f1/iou: "
      f"{test_macro['precision']:.4f}/{test_macro['recall']:.4f}/"
      f"{test_macro['f1']:.4f}/{test_macro['iou']:.4f}")

# ============================================================
# ORACLE BOUND
# ============================================================
# The best F1 test itself could reach under any threshold -- never selected,
# never reported as the model's number, only computed to show how much of the
# val -> test gap is threshold transfer (closed by picking the test-optimal
# threshold) versus domain shift (whatever gap remains at the oracle).

test_oracle_row = max(test_rows, key=lambda row: row['f1'])
test_oracle = {
    'threshold': test_oracle_row['threshold'],
    'f1': test_oracle_row['f1'],
    'gap_vs_selected': test_oracle_row['f1'] - final_test['f1'],
}
print(
    f"test F1 @ val-selected thr {final_threshold:.3f}: {final_test['f1']:.4f}  |  "
    f"@ test-oracle thr {test_oracle['threshold']:.3f}: {test_oracle['f1']:.4f}  "
    "(bound, not reported)"
)

# Same weights under the cheap training-time protocol, so the gain from D4 TTA
# is attributable rather than assumed. This block stays on the plain
# sweep_thresholds/evaluate_threshold path (not the cubes above) on purpose:
# it exists to hold the eval protocol as the only variable between the two
# numbers, not to be selected against.
quick_threshold, quick_val_m, _ = sweep_thresholds(
    final_model, qg_val_ds, QG_EVAL.thresholds, QG_EVAL, CONTEXT,
    amp=True, amp_dtype='float16')
quick_test = evaluate_threshold(
    final_model, qg_test_ds, quick_threshold, QG_EVAL, CONTEXT,
    amp=True, amp_dtype='float16', final=False)

# Same weights, same SELECTED threshold (final_threshold), cheap protocol
# (no TTA): this is the actual protocol-only comparison against final_test
# -- quick_test above is evaluated at quick_threshold (argmaxed on the
# PRUNED val over QG_EVAL.thresholds), so quick_test vs final_test differs
# in TTA, threshold rule AND selection set, not TTA alone.
test_no_tta_at_selected_threshold = evaluate_threshold(
    final_model, qg_test_ds, final_threshold, QG_EVAL, CONTEXT,
    amp=True, amp_dtype='float16', final=False)

final_report = {
    'dataset': 'ValaisCD',
    'version': VALAIS_VERSION or 'full_unpruned',
    'weights_source': FINAL_SOURCE,
    'checkpoint': str(QG_BEST_PATH),
    'best_epoch': qg_best.get('best_epoch'),
    'selected_weights': SELECTED,
    'threshold': final_threshold,
    'val': final_val,
    'test': final_test,
    'val_macro': val_macro,
    'test_macro': test_macro,
    'test_oracle': test_oracle,
    'protocol': {
        'train_city': 'Sion',
        'val_city': 'Sierre',
        'test_city': 'Martigny',
        'selection_set': 'val_unpruned_1716',
        'reported_val_set': 'val_pruned_500',
        'threshold_rule': 'plateau_median_within_1pct',
    },
    'quick_protocol': {
        'val_note': 'self-selected: evaluated at quick_threshold, its own argmax on this same pruned val -- optimistic, compare val / val_macro above instead',
        'threshold': quick_threshold,
        'val': quick_val_m,
        'test': quick_test,
    },
    'test_no_tta_at_selected_threshold': test_no_tta_at_selected_threshold,
    'protocol_gain_test_f1': final_test['f1'] - test_no_tta_at_selected_threshold['f1'],
}

print(
    f"\nprotocol gain on test F1 (same threshold {final_threshold:.3f}, same selection set, TTA only): "
    f"{test_no_tta_at_selected_threshold['f1']:.4f} (no TTA)"
    f" -> {final_test['f1']:.4f} (D4 TTA)"
    f"  = {final_test['f1'] - test_no_tta_at_selected_threshold['f1']:+.4f}"
)
print(json.dumps(final_report, indent=2))
with open(OUT_DIR / 'qgmamba_valais_final_metrics.json', 'w') as handle:
    json.dump(final_report, handle, indent=2)

# zero-shot vs fine-tuned comparison, both on the ValaisCD test split. The
# filename here is the one cell B actually writes.
try:
    with open(OUT_DIR / 'qgmamba_valais_zero_shot_metrics.json') as handle:
        zs_test = json.load(handle)['test_at_swept_threshold']
    comparison = pd.DataFrame([
        {'model': 'QGMamba zero-shot (S2Looking ckpt)',
         **{k: zs_test[k] for k in ('precision', 'recall', 'f1', 'iou')}},
        {'model': 'QGMamba fine-tuned on ValaisCD',
         **{k: final_test[k] for k in ('precision', 'recall', 'f1', 'iou')}},
    ])
    print(comparison.to_string(index=False))
except FileNotFoundError:
    pass

# training curves (from the checkpoint, so this cell also works standalone)
curves = qg_best.get('history') or {}
if curves.get('epoch'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(curves['epoch'], curves['train_loss'])
    axes[0].set_title('QGMamba train loss'); axes[0].set_xlabel('epoch')
    axes[0].grid(alpha=0.3)
    axes[1].plot(curves['epoch'], curves['val_iou'], label='val IoU')
    axes[1].plot(curves['epoch'], curves['val_f1'], label='val F1')
    axes[1].set_title('QGMamba validation on ValaisCD'); axes[1].set_xlabel('epoch')
    axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print('\nno training history in this checkpoint, skipping curves')

# final_threshold was calibrated WITH FINAL_EVAL's TTA (D4 TTA on) --
# render the panel at that same protocol, or the shown predictions are
# from a different operating point than the threshold printed under them.
qg_show(final_model, qg_test_ds, final_threshold, n=4, title='ValaisCD',
        tta=FINAL_EVAL.tta, d4_tta=FINAL_EVAL.d4_tta)


In [ ]:
import cv2, numpy as np
from collections import defaultdict

# ============================================================
# SIZE-BINNED COMPONENT RECALL, VAL + TEST, AT THE REPORTED OPERATING POINT
# ============================================================
# For every ground-truth change component, count it as "hit" if the final
# prediction overlaps it by at least one pixel. This measures detection
# recall as a function of object size, which on ValaisCD is where the model
# actually loses: the labels are SwissTLM3D building-footprint differences,
# so a large share of the components are single small structures.
#
# Runs on both val (Sierre) and test (Martigny) so a low bin can be read as
# a size effect (shows up on both splits) or a city effect (test only) --
# cell D already reports a val -> test gap, this is where it is diagnosed.
#
# Inference uses FINAL_EVAL's protocol (same tile size/stride/batch/TTA cell D
# calibrated final_threshold under), not the cheap QG_EVAL protocol: scoring
# at final_threshold under a different protocol than the one that picked it
# is a mismatched operating point.

# PRECONDITION CHECK
# Needs the setup cell (cell A) for qg_val_ds / qg_test_ds /
# predict_probability_map, and the final evaluation cell (cell D) for
# FINAL_EVAL / final_model / final_threshold.

_REQUIRED = [
    "qg_val_ds",
    "qg_test_ds",
    "predict_probability_map",
    "FINAL_EVAL",
    "DEVICE",
    "final_model",
    "final_threshold",
]

_missing = [
    x for x in _REQUIRED
    if x not in globals()
]

if _missing:
    raise RuntimeError(
        "Run the setup cell (cell A) and the final evaluation cell (cell D) "
        "before this one.\n"
        f"Missing: {_missing}"
    )

# GT component area in pixels on the native 256x256 masks. ValaisCD is
# 0.5 m/px, so 1 px = 0.25 m2 and the bin edges below are ~12 m2, ~50 m2
# and ~200 m2: a garage, a house, a warehouse. The S2Looking bins started
# at 200 px, which on this dataset would drop most real footprints into a
# single bucket.
BINS = [(0, 50), (50, 200), (200, 800), (800, 10**9)]


def _component_recall(dataset, name):
    """Hit/total GT components and positive-pixel mass per BINS bin, at
    final_threshold under FINAL_EVAL's protocol. Prints one table for
    `dataset` and returns (hit, tot) keyed by bin, for the comparison below."""

    hit, tot, pixels = defaultdict(int), defaultdict(int), defaultdict(int)
    total_pixels = 0

    for i in tqdm(range(len(dataset)), desc=f"component recall ({name})"):
        image_a, image_b, target, _name = dataset[i]
        prob = predict_probability_map(
            final_model, image_a, image_b, DEVICE,
            FINAL_EVAL.tile_size, FINAL_EVAL.tile_stride, FINAL_EVAL.batch_tiles,
            FINAL_EVAL.tta, True, 'float16', False, FINAL_EVAL.d4_tta)
        pred = (prob > final_threshold).astype(np.uint8)
        n, labels, stats, _ = cv2.connectedComponentsWithStats(
            target.astype(np.uint8), 8)
        for c in range(1, n):                      # skip background (label 0)
            area = int(stats[c, cv2.CC_STAT_AREA])
            lo, hi = next(b for b in BINS if b[0] <= area < b[1])
            tot[(lo, hi)] += 1
            pixels[(lo, hi)] += area
            total_pixels += area
            if pred[labels == c].any():             # any overlap counts as a hit
                hit[(lo, hi)] += 1

    print(f"\ncomponent recall by GT area bin (px) -- {name}:")
    for (lo, hi) in BINS:
        if tot[(lo, hi)] == 0:
            continue
        share = pixels[(lo, hi)] / total_pixels if total_pixels else 0.0
        print(f"  [{lo:>5d}, {hi if hi < 10**9 else 'inf'}): "
              f"{hit[(lo, hi)]}/{tot[(lo, hi)]} = "
              f"{hit[(lo, hi)] / tot[(lo, hi)]:.4f}  "
              f"(pos-pixel share {share:.4f})")

    return hit, tot


val_hit, val_tot = _component_recall(qg_val_ds, 'val (Sierre)')
test_hit, test_tot = _component_recall(qg_test_ds, 'test (Martigny)')

print("\ncomponent recall by GT area bin (px) -- val vs test:")
for (lo, hi) in BINS:
    if val_tot[(lo, hi)] == 0 and test_tot[(lo, hi)] == 0:
        continue
    v = val_hit[(lo, hi)] / val_tot[(lo, hi)] if val_tot[(lo, hi)] else float('nan')
    t = test_hit[(lo, hi)] / test_tot[(lo, hi)] if test_tot[(lo, hi)] else float('nan')
    print(f"  [{lo:>5d}, {hi if hi < 10**9 else 'inf'}): "
          f"val {v:.4f} ({val_hit[(lo, hi)]}/{val_tot[(lo, hi)]})  "
          f"test {t:.4f} ({test_hit[(lo, hi)]}/{test_tot[(lo, hi)]})  "
          f"diff {t - v:+.4f}")


In [ ]:
# ===================== QGMamba-CD: efficiency metrics (cell E) =====================
# Parameters, MACs / FLOPs, latency, throughput and peak memory for the QGMamba
# model, plus what the two evaluation protocols used in cells A and D actually
# cost per full-resolution image.
#
# Three things to know before reading the numbers:
#
#   1. In eval mode the forward pass runs the diffusion sampler for
#      model.diffusion.inference_steps steps (5 under
#      valais_cd_transfer.yaml), so the per-tile cost measured here IS the
#      inference cost. The single-step training forward, and its backward, are
#      reported separately.
#   2. FLOPs are reported as 2 x MACs. Convolutions, matmuls and attention are
#      counted; norms, activations and sigmoids are not. That is the usual
#      convention, and it is why such a number is only meaningful against
#      another number produced the same way.
#   3. Params and FLOPs follow from the config, not from the weights, so this
#      cell is valid even with a freshly built (random) model.

import copy
import gc
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch


# ============================================================
# PRECONDITION CHECK
# ============================================================

_REQUIRED = [
    "build_qgmamba",
    "QG_TILE",
    "QG_EVAL",
    "DEVICE",
    "OUT_DIR",
]

_missing = [x for x in _REQUIRED if x not in globals()]

if _missing:
    raise RuntimeError(
        "Run the QGMamba setup cell (cell A) before this one.\n"
        f"Missing: {_missing}"
    )


# ============================================================
# KNOBS
# ============================================================

EFF_TILE = int(QG_TILE)                     # profile at the tile size used at eval
EFF_WARMUP = 5                              # untimed iterations before each timing run
EFF_ITERS = 20                              # timed iterations at batch 1, scaled down above that
EFF_BATCHES = sorted({1, 4, int(QG_EVAL.batch_tiles)})
EFF_END_TO_END_IMAGES = 3                   # real test images timed per protocol, 0 = skip
EFF_TRAIN_STEP_BATCH = int(globals().get("QG_BATCH", 8))
# Off by default: the smp U-Net baseline needs `smp`, ENCODER and
# NUM_CLASSES, which this ValaisCD notebook never defines (they came from a
# separate U-Net notebook). The block below is inside a try/except, so
# leaving it True would just print a swallowed failure.
INCLUDE_UNET_BASELINE = False

EFF_AMP = torch.cuda.is_available()

EFF_AMP_DTYPE = (
    torch.bfloat16
    if EFF_AMP and torch.cuda.get_device_capability()[0] >= 8
    else torch.float16
)

EFF_AMP_NAME = (
    "bfloat16"
    if EFF_AMP_DTYPE is torch.bfloat16
    else "float16"
)


# ============================================================
# 1. MODEL UNDER TEST
# ============================================================
# Reuse whatever is already in memory so the reported numbers belong to the
# model the notebook just evaluated. Falling back to a fresh build is fine for
# params/FLOPs and for timing -- none of them depend on the weight values.

eff_model = None
eff_source = None

for _name in ("final_model", "swa_model", "zs_model", "qg_model"):

    _candidate = globals().get(_name)

    if isinstance(_candidate, torch.nn.Module):
        eff_model, eff_source = _candidate, _name
        break

if eff_model is None:
    eff_model = build_qgmamba(pretrained=False)
    eff_source = "build_qgmamba(pretrained=False)  [random weights]"

eff_model = eff_model.to(DEVICE).eval()

_diffusion = getattr(eff_model, "diffusion", None)

EFF_DIFFUSION_STEPS = (
    int(getattr(_diffusion, "inference_steps", 0))
    if getattr(eff_model, "diffusion_enabled", False)
    else 0
)

print("=" * 70)
print("EFFICIENCY PROFILE")
print("=" * 70)
print("model under test :", eff_source)
print("encoder          :", getattr(eff_model, "encoder_name", "?"))
print("tile             :", f"{EFF_TILE}x{EFF_TILE}")
print("device           :", DEVICE)

if torch.cuda.is_available():
    print("gpu              :", torch.cuda.get_device_name(0))

print("torch            :", torch.__version__)
print("amp              :", EFF_AMP_NAME if EFF_AMP else "off (fp32)")
print(
    "diffusion        :",
    f"{EFF_DIFFUSION_STEPS} sampler steps at inference"
    if EFF_DIFFUSION_STEPS
    else "disabled"
)


# ============================================================
# 2. PARAMETER COUNTS
# ============================================================

def parameter_counts(module):
    """(total, trainable, buffers) element counts for a module."""

    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    buffers = sum(b.numel() for b in module.buffers())
    return total, trainable, buffers


eff_params, eff_trainable, eff_buffers = parameter_counts(eff_model)

param_rows = []

for name, child in eff_model.named_children():

    child_total, _, _ = parameter_counts(child)

    if child_total:
        param_rows.append({
            "module": name,
            "params_m": child_total / 1.0e6,
            "share_pct": 100.0 * child_total / max(eff_params, 1),
        })

param_table = (
    pd.DataFrame(param_rows)
    .sort_values("params_m", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "-" * 70)
print("PARAMETERS")
print("-" * 70)
print(f"total      : {eff_params / 1.0e6:9.3f} M")
print(f"trainable  : {eff_trainable / 1.0e6:9.3f} M")
print(f"buffers    : {eff_buffers / 1.0e6:9.3f} M  (not optimised)")
print(f"fp32 size  : {eff_params * 4 / 2 ** 20:9.1f} MiB")
print(f"fp16 size  : {eff_params * 2 / 2 ** 20:9.1f} MiB")

_ckpt_path = globals().get("QG_BEST_PATH")

if _ckpt_path is not None and Path(_ckpt_path).exists():
    print(
        f"checkpoint : {Path(_ckpt_path).stat().st_size / 2 ** 20:9.1f} MiB"
        "  (weights + EMA + optimizer + scheduler)"
    )

print()
print(param_table.to_string(index=False, float_format=lambda v: f"{v:8.3f}"))


# ============================================================
# 3. FLOP COUNTING
# ============================================================
# Three backends, tried in order. The first needs no external package; the
# other two are there so the cell still reports something on older torch.

def _flops_native(model, inputs, backward_from=None):
    """torch.utils.flop_counter -- real FLOPs, with per-module attribution."""

    from torch.utils.flop_counter import FlopCounterMode

    counter = FlopCounterMode(display=False)

    if backward_from is None:
        with counter, torch.no_grad():
            model(*inputs)
    else:
        with counter:
            backward_from(model(*inputs)).backward()

    per_module = {}

    for name, operations in counter.get_flop_counts().items():

        parts = name.split(".")

        if len(parts) == 2:                 # 'QGMambaDiffCD.encoder' -> 'encoder'
            per_module[parts[1]] = float(sum(operations.values()))

    return float(counter.get_total_flops()), per_module, "torch.utils.flop_counter"


def _flops_fvcore(model, inputs, backward_from=None):

    if backward_from is not None:
        raise NotImplementedError("fvcore counts the forward pass only")

    from fvcore.nn import FlopCountAnalysis

    analysis = FlopCountAnalysis(model, inputs)
    analysis.unsupported_ops_warnings(False)
    analysis.uncalled_modules_warnings(False)

    per_module = {
        name: float(value) * 2.0
        for name, value in analysis.by_module().items()
        if name and "." not in name
    }

    return float(analysis.total()) * 2.0, per_module, "fvcore (MACs x 2)"


def _flops_thop(model, inputs, backward_from=None):

    if backward_from is not None:
        raise NotImplementedError("thop counts the forward pass only")

    import thop

    # thop attaches total_ops / total_params buffers to every submodule -- the
    # exact pollution qgmamba_cd.checkpoint.normalize_state_dict strips out on
    # load. Profile a copy so the live model keeps a clean state_dict.
    clone = copy.deepcopy(model)
    macs, _ = thop.profile(clone, inputs=inputs, verbose=False)
    del clone

    return float(macs) * 2.0, {}, "thop (MACs x 2)"


def measure_flops(model, inputs, backward_from=None):
    """Returns (flops, per_module_flops, backend_name). NaN if none available."""

    errors = []

    for backend in (_flops_native, _flops_fvcore, _flops_thop):

        try:
            return backend(model, inputs, backward_from)

        except Exception as error:
            errors.append(f"{backend.__name__}: {type(error).__name__}: {error}")

    print("\nFLOP counting unavailable (pip install fvcore, or upgrade torch):")

    for line in errors:
        print("  ", line)

    return float("nan"), {}, "unavailable"


_probe_a = torch.randn(1, 3, EFF_TILE, EFF_TILE, device=DEVICE)
_probe_b = torch.randn_like(_probe_a)

eff_flops, eff_flops_by_module, eff_flop_backend = measure_flops(
    eff_model, (_probe_a, _probe_b)
)

del _probe_a, _probe_b

print("\n" + "-" * 70)
print("FLOPs / MACs PER TILE  (batch 1, inference path)")
print("-" * 70)
print("backend     :", eff_flop_backend)
print(f"GFLOPs      : {eff_flops / 1.0e9:9.2f}")
print(f"GMACs       : {eff_flops / 2.0e9:9.2f}")

if eff_params:
    print(
        f"intensity   : {eff_flops / 1.0e9 / (eff_params / 1.0e6):9.2f}"
        " GFLOPs per M params"
    )

flop_table = pd.DataFrame()

if eff_flops_by_module:

    flop_table = (
        pd.DataFrame([
            {
                "module": name,
                "gflops": value / 1.0e9,
                "share_pct": 100.0 * value / max(eff_flops, 1.0),
            }
            for name, value in eff_flops_by_module.items()
            if value > 0
        ])
        .sort_values("gflops", ascending=False)
        .reset_index(drop=True)
    )

    print()
    print(flop_table.to_string(index=False, float_format=lambda v: f"{v:8.2f}"))


# ============================================================
# 4. TRAINING-STEP COST (forward + backward, one diffusion step)
# ============================================================

train_flops = float("nan")
train_peak_mib = float("nan")
train_backend = "skipped"

try:
    eff_model.train()

    _train_a = torch.randn(EFF_TRAIN_STEP_BATCH, 3, EFF_TILE, EFF_TILE, device=DEVICE)
    _train_b = torch.randn_like(_train_a)
    _train_gt = (
        torch.rand(EFF_TRAIN_STEP_BATCH, 1, EFF_TILE, EFF_TILE, device=DEVICE) > 0.9
    ).float()

    # Surrogate scalar in place of FullLoss: the loss itself is a rounding error
    # next to the network, and this keeps the measurement independent of the
    # loss weights in the YAML.
    train_flops, _, train_backend = measure_flops(
        eff_model,
        (_train_a, _train_b, _train_gt),
        backward_from=lambda outputs: outputs[0].float().sum(),
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    with torch.autocast("cuda", dtype=EFF_AMP_DTYPE, enabled=EFF_AMP):
        _outputs = eff_model(_train_a, _train_b, _train_gt)
        _outputs[0].float().sum().backward()

    if torch.cuda.is_available():
        torch.cuda.synchronize()
        train_peak_mib = torch.cuda.max_memory_allocated() / 2 ** 20

    print("\n" + "-" * 70)
    print(f"TRAINING STEP  (batch {EFF_TRAIN_STEP_BATCH}, fwd + bwd)")
    print("-" * 70)
    print(f"TFLOPs/step  : {train_flops / 1.0e12:9.3f}   [{train_backend}]")
    print(f"GFLOPs/sample: {train_flops / 1.0e9 / EFF_TRAIN_STEP_BATCH:9.2f}")
    print(f"peak memory  : {train_peak_mib:9.1f} MiB  (activations + grads, no optimizer state)")
    print(
        f"optimizer    : {eff_params * 8 / 2 ** 20:9.1f} MiB"
        "  (AdamW exp_avg + exp_avg_sq, fp32)"
    )
    print(f"EMA shadow   : {eff_params * 4 / 2 ** 20:9.1f} MiB")

except Exception as error:
    print("\ntraining-step profile skipped:", type(error).__name__, error)

finally:
    # A backward on the live model leaves .grad populated on every parameter,
    # which would otherwise be picked up by the next optimizer step in cell C.
    eff_model.zero_grad(set_to_none=True)
    eff_model.eval()

    for _name in ("_train_a", "_train_b", "_train_gt", "_outputs"):
        globals().pop(_name, None)

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# 5. LATENCY / THROUGHPUT / PEAK MEMORY
# ============================================================

def benchmark(run, batch, iters, warmup=EFF_WARMUP):
    """Mean wall clock of run(), with steady-state peak allocation."""

    with torch.inference_mode():

        for _ in range(warmup):
            run()

        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()

        start = time.perf_counter()

        for _ in range(iters):
            run()

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - start

    milliseconds = elapsed / iters * 1000.0

    return {
        "batch": batch,
        "iters": iters,
        "latency_ms": milliseconds,
        "ms_per_tile": milliseconds / batch,
        "tiles_per_s": batch * iters / elapsed,
        "peak_mem_mib": (
            torch.cuda.max_memory_allocated() / 2 ** 20
            if torch.cuda.is_available()
            else float("nan")
        ),
    }


def make_runner(model, batch, stacked=False):
    """stacked=True feeds one 6-channel tensor (the smp U-Net convention)."""

    image_a = torch.randn(batch, 3, EFF_TILE, EFF_TILE, device=DEVICE)
    image_b = torch.randn_like(image_a)
    stack = torch.cat([image_a, image_b], dim=1) if stacked else None

    def run():
        with torch.autocast("cuda", dtype=EFF_AMP_DTYPE, enabled=EFF_AMP):
            model(stack) if stacked else model(image_a, image_b)

    return run


timing_rows = []

for _batch in EFF_BATCHES:

    _iters = max(3, round(EFF_ITERS / max(1.0, _batch / 2.0)))

    try:
        timing_rows.append(
            benchmark(make_runner(eff_model, _batch), _batch, _iters)
        )

    except Exception as error:
        print(f"batch {_batch} timing skipped:", type(error).__name__, error)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

timing_table = pd.DataFrame(timing_rows)

print("\n" + "-" * 70)
print(f"LATENCY / THROUGHPUT  ({EFF_TILE}x{EFF_TILE} tiles, {EFF_AMP_NAME if EFF_AMP else 'fp32'})")
print("-" * 70)
print(timing_table.to_string(index=False, float_format=lambda v: f"{v:9.2f}"))


# ============================================================
# 6. WHAT A FULL IMAGE COSTS UNDER EACH PROTOCOL
# ============================================================
# The metrics cells never run the model on a single tile: they tile the full
# image and, for the final protocol, average several TTA views. Per-tile numbers
# understate the real cost by that multiplier, so derive it from the repo's own
# tiling function and the EvalConfig actually in use.

from qgmamba_cd.evaluation import tile_coordinates

# ValaisCD patches are 256x256, i.e. exactly one model tile, so a "full
# image" costs one forward pass per TTA view rather than a grid of tiles.
# The probe below overrides this from the actual test set anyway.
FULL_H = FULL_W = 256

if "qg_test_ds" in globals() and len(qg_test_ds) > 0:
    _shape_probe = qg_test_ds[0][0]
    FULL_H, FULL_W = int(_shape_probe.shape[-2]), int(_shape_probe.shape[-1])
    del _shape_probe


def protocol_passes(evaluation, final):
    """(tiles, tta_views, forward_passes) per full image, mirroring evaluation.py."""

    stride = evaluation.final_stride if final else evaluation.tile_stride
    tta = evaluation.final_tta if final else evaluation.tta

    views = (8 if evaluation.d4_tta else 4) if tta else 1

    ys, xs = tile_coordinates(FULL_H, FULL_W, evaluation.tile_size, stride)
    tiles = len(ys) * len(xs)

    return tiles, views, tiles * views


_ms_per_tile_at_batch = {row["batch"]: row["ms_per_tile"] for row in timing_rows}

protocol_rows = []

_protocols = [
    ("quick  (cell A / per-epoch val)", QG_EVAL, False),
    ("final  (cell D / reported test)", globals().get("FINAL_EVAL", QG_EVAL), True),
]

for _label, _evaluation, _final in _protocols:

    _tiles, _views, _passes = protocol_passes(_evaluation, _final)

    _ms_per_tile = _ms_per_tile_at_batch.get(
        int(_evaluation.batch_tiles),
        min(_ms_per_tile_at_batch.values()) if _ms_per_tile_at_batch else float("nan"),
    )

    protocol_rows.append({
        "protocol": _label,
        "stride": _evaluation.final_stride if _final else _evaluation.tile_stride,
        "tiles": _tiles,
        "tta_views": _views,
        "fwd_passes": _passes,
        "gflops_per_image": eff_flops * _passes / 1.0e9,
        "est_ms_per_image": _ms_per_tile * _passes,
        "est_images_per_s": 1000.0 / max(_ms_per_tile * _passes, 1.0e-9),
    })

protocol_table = pd.DataFrame(protocol_rows)

print("\n" + "-" * 70)
print(f"COST PER {FULL_H}x{FULL_W} IMAGE")
print("-" * 70)
print(protocol_table.to_string(index=False, float_format=lambda v: f"{v:10.2f}"))


# ============================================================
# 7. MEASURED END-TO-END COST ON REAL TEST IMAGES
# ============================================================
# The estimate above multiplies a synthetic per-tile time by the pass count. It
# ignores the accumulator arithmetic, the host-device copies and the batching
# remainder, so measure the repo's predict_probability_map directly as a check.

end_to_end = {}

if (
    EFF_END_TO_END_IMAGES
    and "qg_test_ds" in globals()
    and "predict_probability_map" in globals()
    and len(qg_test_ds) > 0
):

    print("\n" + "-" * 70)
    print(f"MEASURED END-TO-END  ({min(EFF_END_TO_END_IMAGES, len(qg_test_ds))} test images)")
    print("-" * 70)

    for _label, _evaluation, _final in _protocols:

        _stride = _evaluation.final_stride if _final else _evaluation.tile_stride
        _tta = _evaluation.final_tta if _final else _evaluation.tta

        _durations = []

        for _index in range(min(EFF_END_TO_END_IMAGES, len(qg_test_ds))):

            _image_a, _image_b, _, _ = qg_test_ds[_index]

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            _start = time.perf_counter()

            predict_probability_map(
                eff_model,
                _image_a,
                _image_b,
                DEVICE,
                _evaluation.tile_size,
                _stride,
                _evaluation.batch_tiles,
                _tta,
                EFF_AMP,
                EFF_AMP_NAME,
                False,
                _evaluation.d4_tta,
            )

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            _durations.append(time.perf_counter() - _start)

        # First image absorbs the kernel autotuning for this tile/stride combination.
        _steady = _durations[1:] or _durations

        end_to_end[_label.split()[0]] = {
            "ms_per_image": float(np.mean(_steady) * 1000.0),
            "images_per_s": float(1.0 / np.mean(_steady)),
            "first_image_ms": float(_durations[0] * 1000.0),
        }

        print(
            f"{_label:34s} "
            f"{np.mean(_steady) * 1000.0:9.1f} ms/image   "
            f"{1.0 / np.mean(_steady):6.2f} images/s"
        )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# 8. U-NET BASELINE (same tile, same device)
# ============================================================

baseline = {}

if INCLUDE_UNET_BASELINE:

    try:
        _unet = smp.Unet(
            encoder_name=ENCODER,
            encoder_weights=None,
            in_channels=6,
            classes=NUM_CLASSES,
        ).to(DEVICE).eval()

        _unet_params, _, _ = parameter_counts(_unet)

        _unet_flops, _, _unet_backend = measure_flops(
            _unet, (torch.randn(1, 6, EFF_TILE, EFF_TILE, device=DEVICE),)
        )

        _unet_timing = benchmark(
            make_runner(_unet, 1, stacked=True), 1, EFF_ITERS
        )

        baseline = {
            "name": f"U-Net ({ENCODER}, 6-channel early fusion)",
            "params_m": _unet_params / 1.0e6,
            "gflops_per_tile": _unet_flops / 1.0e9,
            "ms_per_tile": _unet_timing["ms_per_tile"],
            "flop_backend": _unet_backend,
        }

        print("\n" + "-" * 70)
        print("BASELINE COMPARISON  (per tile, batch 1)")
        print("-" * 70)

        print(
            pd.DataFrame([
                {
                    "model": baseline["name"],
                    "params_m": baseline["params_m"],
                    "gflops": baseline["gflops_per_tile"],
                    "ms": baseline["ms_per_tile"],
                },
                {
                    "model": "QGMamba-CD (siamese + diffusion refinement)",
                    "params_m": eff_params / 1.0e6,
                    "gflops": eff_flops / 1.0e9,
                    "ms": _ms_per_tile_at_batch.get(1, float("nan")),
                },
            ]).to_string(index=False, float_format=lambda v: f"{v:9.2f}")
        )

        del _unet
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as error:
        print("\nU-Net baseline skipped:", type(error).__name__, error)


# ============================================================
# 9. SAVE
# ============================================================

efficiency_report = {

    "model_source": eff_source,
    "encoder": getattr(eff_model, "encoder_name", None),
    "torch_version": torch.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",

    "setup": {
        "tile": EFF_TILE,
        "full_image": [FULL_H, FULL_W],
        "amp": EFF_AMP_NAME if EFF_AMP else "fp32",
        "diffusion_infer_steps": EFF_DIFFUSION_STEPS,
        "flop_backend": eff_flop_backend,
        "flops_convention": "2 x MACs; norms/activations excluded",
    },

    "parameters": {
        "total_m": eff_params / 1.0e6,
        "trainable_m": eff_trainable / 1.0e6,
        "buffers_m": eff_buffers / 1.0e6,
        "fp32_mib": eff_params * 4 / 2 ** 20,
        "fp16_mib": eff_params * 2 / 2 ** 20,
        "by_module_m": {
            row["module"]: row["params_m"]
            for row in param_table.to_dict("records")
        },
    },

    "inference_per_tile": {
        "gflops": eff_flops / 1.0e9,
        "gmacs": eff_flops / 2.0e9,
        "by_module_gflops": {
            name: value / 1.0e9 for name, value in eff_flops_by_module.items()
        },
        "timing": timing_rows,
    },

    "training_step": {
        "batch": EFF_TRAIN_STEP_BATCH,
        "tflops_fwd_bwd": train_flops / 1.0e12,
        "gflops_per_sample": (
            train_flops / 1.0e9 / EFF_TRAIN_STEP_BATCH
            if EFF_TRAIN_STEP_BATCH else float("nan")
        ),
        "peak_activation_mib": train_peak_mib,
        "optimizer_state_mib": eff_params * 8 / 2 ** 20,
        "ema_shadow_mib": eff_params * 4 / 2 ** 20,
        "backend": train_backend,
    },

    "per_image_protocols": protocol_table.to_dict("records"),
    "measured_end_to_end": end_to_end,
    "baseline": baseline,
}

_efficiency_path = OUT_DIR / "qgmamba_efficiency_metrics.json"

OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(_efficiency_path, "w") as handle:
    json.dump(efficiency_report, handle, indent=2, default=float)

print("\n" + "=" * 70)
print("HEADLINE")
print("=" * 70)
print(f"params            : {eff_params / 1.0e6:.2f} M")
print(f"GFLOPs / tile     : {eff_flops / 1.0e9:.2f}   ({EFF_TILE}x{EFF_TILE}, {EFF_DIFFUSION_STEPS} diffusion steps)")

for _row in protocol_rows:
    print(
        f"GFLOPs / image    : {_row['gflops_per_image']:.1f}"
        f"   [{_row['protocol'].split()[0]}, {_row['fwd_passes']} forward passes]"
    )

if _ms_per_tile_at_batch:
    _largest_batch = max(_ms_per_tile_at_batch)
    print(
        f"ms / tile (b={_largest_batch:<2d}) : "
        f"{_ms_per_tile_at_batch[_largest_batch]:.2f}"
    )

if not timing_table.empty:
    print(f"peak inference mem: {timing_table['peak_mem_mib'].max():.0f} MiB")

print("\n✓ written to", _efficiency_path)